In [ ]:
# ============================================================
# AMLC 2026 — CELL 01
# Runtime diagnostics
# ============================================================

import os
import sys
import platform
import shutil
import subprocess

print("=" * 70)
print("AMLC 2026 ENTITY RESOLUTION — RUNTIME CHECK")
print("=" * 70)

print("\nPython:")
print(sys.version)

print("\nPlatform:")
print(platform.platform())

print("\nCPU:")
try:
    print(subprocess.check_output(["bash", "-lc", "nproc"]).decode().strip())
except Exception:
    print("Could not determine CPU count")

print("\nRAM:")
try:
    print(subprocess.check_output(
        ["bash", "-lc", "free -h"]
    ).decode())
except Exception:
    print("Could not determine RAM")

print("GPU:")
try:
    print(subprocess.check_output(
        ["bash", "-lc", "nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader"]
    ).decode())
except Exception as e:
    print("No GPU detected:", e)

print("\nDisk:")
try:
    print(subprocess.check_output(
        ["bash", "-lc", "df -h /content"]
    ).decode())
except Exception:
    print("Could not determine disk")

print("\nWorking directory:")
print(os.getcwd())

print("=" * 70)

AMLC 2026 ENTITY RESOLUTION — RUNTIME CHECK

Python:
3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]

Platform:
Linux-6.6.122+-x86_64-with-glibc2.39

CPU:
2

RAM:
               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.1Gi       8.2Gi       2.2Mi       3.6Gi        11Gi
Swap:             0B          0B          0B

GPU:
Tesla T4, 15360 MiB, 580.82.07


Disk:
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   48G   66G  43% /


Working directory:
/content


In [5]:
# ============================================================
# AMLC 2026 — CELL 02
# Mount Google Drive
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# ============================================================
# AMLC 2026 — CELL 03
# Verify project structure
# ============================================================

from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/AMLC2026")

DATA_ROOT = DRIVE_ROOT / "dataset"
TRAIN_ROOT = DATA_ROOT / "train"
TEST_ROOT = DATA_ROOT / "test"

print("DRIVE_ROOT:", DRIVE_ROOT)
print("DATA_ROOT :", DATA_ROOT)
print("TRAIN_ROOT:", TRAIN_ROOT)
print("TEST_ROOT :", TEST_ROOT)

print("\nTrain files:")
for p in sorted(TRAIN_ROOT.glob("*")):
    print(f"{p.name:30s} {p.stat().st_size / (1024**3):.2f} GB")

print("\nTest files:")
for p in sorted(TEST_ROOT.glob("*")):
    print(f"{p.name:30s} {p.stat().st_size / (1024**3):.2f} GB")

DRIVE_ROOT: /content/drive/MyDrive/AMLC2026
DATA_ROOT : /content/drive/MyDrive/AMLC2026/dataset
TRAIN_ROOT: /content/drive/MyDrive/AMLC2026/dataset/train
TEST_ROOT : /content/drive/MyDrive/AMLC2026/dataset/test

Train files:
train_ground_truth.tsv         0.12 GB
train_source1.tsv              0.20 GB
train_source2.tsv              0.46 GB
train_source3.tsv              0.47 GB

Test files:
test_source1.tsv               0.16 GB
test_source2.tsv               0.47 GB
test_source3.tsv               0.47 GB


In [ ]:
# ============================================================
# AMLC 2026 — CELL 04
# Local working directories
# ============================================================

from pathlib import Path

LOCAL_ROOT = Path("/content/AMLC2026")

LOCAL_DATA = LOCAL_ROOT / "dataset"
LOCAL_CACHE = LOCAL_ROOT / "cache"
LOCAL_FEATURES = LOCAL_ROOT / "features"
LOCAL_EMBEDDINGS = LOCAL_ROOT / "embeddings"
LOCAL_MODELS = LOCAL_ROOT / "models"
LOCAL_OOF = LOCAL_ROOT / "oof"
LOCAL_SUBMISSIONS = LOCAL_ROOT / "submissions"
LOCAL_LOGS = LOCAL_ROOT / "logs"

for p in [
    LOCAL_DATA,
    LOCAL_CACHE,
    LOCAL_FEATURES,
    LOCAL_EMBEDDINGS,
    LOCAL_MODELS,
    LOCAL_OOF,
    LOCAL_SUBMISSIONS,
    LOCAL_LOGS,
]:
    p.mkdir(parents=True, exist_ok=True)

print("Created local workspace:")
print(LOCAL_ROOT)

!df -h /content

Created local workspace:
/content/AMLC2026
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   48G   66G  43% /


In [2]:
# ============================================================
# AMLC 2026 — CELL 05
# Core data / ER stack
# ============================================================

!pip -q install \
    polars \
    pyarrow \
    duckdb \
    rapidfuzz \
    unidecode \
    faiss-cpu \
    scikit-learn \
    lightgbm \
    xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 63.2 MB/s eta 0:00:00


In [ ]:
# ============================================================
# AMLC 2026 — CELL 06
# Verify packages
# ============================================================

import polars as pl
import pandas as pd
import numpy as np
import duckdb
import rapidfuzz
import sklearn
import lightgbm
import xgboost

print("Polars     :", pl.__version__)
print("Pandas     :", pd.__version__)
print("NumPy      :", np.__version__)
print("DuckDB     :", duckdb.__version__)
print("RapidFuzz  :", rapidfuzz.__version__)
print("Scikit     :", sklearn.__version__)
print("LightGBM   :", lightgbm.__version__)
print("XGBoost    :", xgboost.__version__)

try:
    import faiss
    print("FAISS      :", faiss.__version__)
except Exception as e:
    print("FAISS import error:", e)

Polars     : 1.35.2
Pandas     : 2.2.3
NumPy      : 2.1.3
DuckDB     : 1.3.2
RapidFuzz  : 3.14.6
Scikit     : 1.6.1
LightGBM   : 4.6.0
XGBoost    : 3.4.1
FAISS      : 1.15.1


In [ ]:
# ============================================================
# AMLC 2026 — CELL 07
# Copy raw dataset to local Colab storage
#
# Drive = persistent master
# /content = high-speed working copy
# ============================================================

import subprocess
from pathlib import Path

LOCAL_DATA = Path("/content/AMLC2026/dataset")
LOCAL_DATA.mkdir(parents=True, exist_ok=True)

print("Copying dataset from Google Drive to local storage...")
print("This should only need to happen once per Colab runtime.\n")

cmd = [
    "rsync",
    "-ah",
    "--info=progress2",
    "--exclude=.DS_Store",
    "/content/drive/MyDrive/AMLC2026/dataset/",
    "/content/AMLC2026/dataset/"
]

result = subprocess.run(cmd)

if result.returncode != 0:
    raise RuntimeError("Dataset copy failed.")

print("\n✅ Dataset copy complete.")

Copying dataset from Google Drive to local storage...
This should only need to happen once per Colab runtime.


✅ Dataset copy complete.


In [ ]:
# ============================================================
# AMLC 2026 — CELL 08
# Verify local dataset
# ============================================================

from pathlib import Path

LOCAL_TRAIN = Path("/content/AMLC2026/dataset/train")
LOCAL_TEST  = Path("/content/AMLC2026/dataset/test")

print("TRAIN")
print("-" * 70)

for p in sorted(LOCAL_TRAIN.glob("*")):
    size_gb = p.stat().st_size / (1024 ** 3)
    print(f"{p.name:30s} {size_gb:8.3f} GB")

print("\nTEST")
print("-" * 70)

for p in sorted(LOCAL_TEST.glob("*")):
    size_gb = p.stat().st_size / (1024 ** 3)
    print(f"{p.name:30s} {size_gb:8.3f} GB")

print("\nDisk after copy:")
!df -h /content

TRAIN
----------------------------------------------------------------------
train_ground_truth.tsv            0.118 GB
train_source1.tsv                 0.196 GB
train_source2.tsv                 0.456 GB
train_source3.tsv                 0.469 GB

TEST
----------------------------------------------------------------------
test_source1.tsv                  0.163 GB
test_source2.tsv                  0.474 GB
test_source3.tsv                  0.471 GB

Disk after copy:
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   53G   61G  47% /


In [ ]:
# ============================================================
# AMLC 2026 — CELL 09
# Robust TSV loader + schema inspection
# ============================================================

import polars as pl
from pathlib import Path

TRAIN_DIR = Path("/content/AMLC2026/dataset/train")
TEST_DIR  = Path("/content/AMLC2026/dataset/test")

files = {
    "train_source1": TRAIN_DIR / "train_source1.tsv",
    "train_source2": TRAIN_DIR / "train_source2.tsv",
    "train_source3": TRAIN_DIR / "train_source3.tsv",
    "train_ground_truth": TRAIN_DIR / "train_ground_truth.tsv",
    "test_source1": TEST_DIR / "test_source1.tsv",
    "test_source2": TEST_DIR / "test_source2.tsv",
    "test_source3": TEST_DIR / "test_source3.tsv",
}


def clean_column_name(name: str) -> str:
    """
    Remove BOM / leading / trailing whitespace from column names.
    Raw data is never modified.
    """
    return (
        str(name)
        .replace("\ufeff", "")
        .strip()
    )


def scan_tsv(path: Path) -> pl.LazyFrame:
    """
    Robust lazy TSV scanner for AMLC 2026.
    Normalizes only the column names.
    """
    lf = pl.scan_csv(
        path,
        separator="\t",
        infer_schema_length=1000,
        null_values=["", "null", "NULL", "None"],
    )

    schema = lf.collect_schema()

    rename_map = {
        col: clean_column_name(col)
        for col in schema.names()
        if clean_column_name(col) != col
    }

    if rename_map:
        print(f"⚠️ Header normalization: {path.name}")
        print("   ", rename_map)
        lf = lf.rename(rename_map)

    return lf


print("=" * 90)
print("SCHEMA AUDIT")
print("=" * 90)

for name, path in files.items():

    print("\n" + "=" * 90)
    print(name)
    print("=" * 90)

    lf = scan_tsv(path)

    print(lf.collect_schema())

SCHEMA AUDIT

train_source1
Schema({'entity_id': String, 'business_name': String, 'business_address': String, 'country': String})

train_source2
Schema({'entity_id': String, 'business_name': String, 'business_address': String, 'country': String})

train_source3
Schema({'entity_id': String, 'business_name': String, 'business_address': String, 'country': String})

train_ground_truth
Schema({'source1_entity_id': String, 'matched_entity_ids': String})

test_source1
Schema({'entity_id': String, 'business_name': String, 'business_address': String, 'country': String})

test_source2
Schema({'entity_id': String, 'business_name': String, 'business_address': String, 'country': String})

test_source3
⚠️ Header normalization: test_source3.tsv
    {' entity_id': 'entity_id'}
Schema({'entity_id': String, 'business_name': String, 'business_address': String, 'country': String})


In [ ]:
# ============================================================
# AMLC 2026 — CELL 10
# Exact row counts
# ============================================================

row_counts = {}

for name, path in files.items():

    count = (
        scan_tsv(path)
        .select(pl.len())
        .collect()
        .item()
    )

    row_counts[name] = count

print("=" * 80)
print("ROW COUNTS")
print("=" * 80)

for name, count in row_counts.items():
    print(f"{name:22s}: {count:,}")

print("\n" + "-" * 80)

train_total = sum(
    row_counts[k]
    for k in [
        "train_source1",
        "train_source2",
        "train_source3",
    ]
)

test_total = sum(
    row_counts[k]
    for k in [
        "test_source1",
        "test_source2",
        "test_source3",
    ]
)

print(f"Total train source rows : {train_total:,}")
print(f"Total test source rows  : {test_total:,}")

⚠️ Header normalization: test_source3.tsv
    {' entity_id': 'entity_id'}
ROW COUNTS
train_source1         : 2,206,821
train_source2         : 5,034,616
train_source3         : 5,285,603
train_ground_truth    : 2,206,821
test_source1          : 1,732,544
test_source2          : 4,887,273
test_source3          : 5,082,316

--------------------------------------------------------------------------------
Total train source rows : 12,527,040
Total test source rows  : 11,702,133


In [ ]:
# ============================================================
# AMLC 2026 — CELL 11
# Sample records
# ============================================================

for name, path in files.items():

    print("\n" + "=" * 100)
    print(name)
    print("=" * 100)

    df = (
        pl.scan_csv(
            path,
            separator="\t",
            infer_schema_length=1000,
            null_values=["", "null", "NULL", "None"]
        )
        .head(10)
        .collect()
    )

    print(df)


train_source1
shape: (10, 4)
┌──────────────┬─────────────────────────────────┬─────────────────────────────────┬─────────┐
│ entity_id    ┆ business_name                   ┆ business_address                ┆ country │
│ ---          ┆ ---                             ┆ ---                             ┆ ---     │
│ str          ┆ str                             ┆ str                             ┆ str     │
╞══════════════╪═════════════════════════════════╪═════════════════════════════════╪═════════╡
│ S1-925783039 ┆ Orelee's Barbershop             ┆ 1795 Westchester Drive, High P… ┆ US      │
│ S1-773889195 ┆ Prime Money                     ┆ 17560 Ellis Road, Tahlequah, O… ┆ US      │
│ S1-377745466 ┆ B+ Retail Inc                   ┆ 1712 Montebello Avenue, Phoeni… ┆ US      │
│ S1-133037285 ┆ Christ Chapel                   ┆ 2100 Cameron Drive, Unit APART… ┆ US      │
│ S1-755362802 ┆ Prabhav Business Center         ┆ 797, Lake Town Block A, Kolkat… ┆ India   │
│ S1-851869949 ┆ Cus

In [ ]:
# ============================================================
# AMLC 2026 — CELL 12
# Source-level structural statistics
# ============================================================

source_files = [
    "train_source1",
    "train_source2",
    "train_source3",
    "test_source1",
    "test_source2",
    "test_source3",
]

for name in source_files:

    path = files[name]

    print("\n" + "=" * 90)
    print(name)
    print("=" * 90)

    lf = scan_tsv(path)

    stats = (
        lf.select([
            pl.len().alias("rows"),

            pl.col("entity_id")
              .n_unique()
              .alias("unique_entity_ids"),

            pl.col("business_name")
              .is_null()
              .sum()
              .alias("null_business_name"),

            pl.col("business_address")
              .is_null()
              .sum()
              .alias("null_business_address"),

            pl.col("country")
              .is_null()
              .sum()
              .alias("null_country"),

            pl.col("business_name")
              .str.len_chars()
              .mean()
              .alias("mean_name_chars"),

            pl.col("business_name")
              .str.len_chars()
              .median()
              .alias("median_name_chars"),

            pl.col("business_address")
              .str.len_chars()
              .mean()
              .alias("mean_address_chars"),

            pl.col("business_address")
              .str.len_chars()
              .median()
              .alias("median_address_chars"),

            pl.col("country")
              .n_unique()
              .alias("unique_countries"),
        ])
        .collect()
    )

    print(stats)


train_source1
shape: (1, 10)
┌─────────┬────────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ rows    ┆ unique_ent ┆ null_busi ┆ null_busi ┆ … ┆ median_na ┆ mean_addr ┆ median_ad ┆ unique_co │
│ ---     ┆ ity_ids    ┆ ness_name ┆ ness_addr ┆   ┆ me_chars  ┆ ess_chars ┆ dress_cha ┆ untries   │
│ u32     ┆ ---        ┆ ---       ┆ ess       ┆   ┆ ---       ┆ ---       ┆ rs        ┆ ---       │
│         ┆ u32        ┆ u32       ┆ ---       ┆   ┆ f64       ┆ f64       ┆ ---       ┆ u32       │
│         ┆            ┆           ┆ u32       ┆   ┆           ┆           ┆ f64       ┆           │
╞═════════╪════════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 2206821 ┆ 2206821    ┆ 0         ┆ 0         ┆ … ┆ 24.0      ┆ 52.066213 ┆ 41.0      ┆ 2         │
└─────────┴────────────┴───────────┴───────────┴───┴───────────┴───────────┴───────────┴───────────┘

train_source2
shape: (1, 10)
┌─────────┬────────────┬───────

In [ ]:
# ============================================================
# AMLC 2026 — CELL 13
# Country distribution
# ============================================================

for name in source_files:

    path = files[name]

    lf = scan_tsv(path)

    print("\n" + "=" * 90)
    print(name)
    print("=" * 90)

    result = (
        lf.group_by("country")
          .agg(pl.len().alias("count"))
          .sort("count", descending=True)
          .collect()
    )

    print(result)


train_source1
shape: (2, 2)
┌─────────┬─────────┐
│ country ┆ count   │
│ ---     ┆ ---     │
│ str     ┆ u32     │
╞═════════╪═════════╡
│ US      ┆ 1323633 │
│ India   ┆ 883188  │
└─────────┴─────────┘

train_source2
shape: (2, 2)
┌─────────┬─────────┐
│ country ┆ count   │
│ ---     ┆ ---     │
│ str     ┆ u32     │
╞═════════╪═════════╡
│ US      ┆ 3016817 │
│ India   ┆ 2017799 │
└─────────┴─────────┘

train_source3
shape: (2, 2)
┌─────────┬─────────┐
│ country ┆ count   │
│ ---     ┆ ---     │
│ str     ┆ u32     │
╞═════════╪═════════╡
│ US      ┆ 3170056 │
│ India   ┆ 2115547 │
└─────────┴─────────┘

test_source1
shape: (3, 2)
┌─────────┬────────┐
│ country ┆ count  │
│ ---     ┆ ---    │
│ str     ┆ u32    │
╞═════════╪════════╡
│ India   ┆ 809986 │
│ US      ┆ 663106 │
│ France  ┆ 259452 │
└─────────┴────────┘

test_source2
shape: (3, 2)
┌─────────┬─────────┐
│ country ┆ count   │
│ ---     ┆ ---     │
│ str     ┆ u32     │
╞═════════╪═════════╡
│ India   ┆ 2312565 │
│ US    

In [ ]:
# ============================================================
# AMLC 2026 — CELL 14
# Ground-truth match cardinality
# ============================================================

GT_PATH = TRAIN_DIR / "train_ground_truth.tsv"

gt = pl.read_csv(
    GT_PATH,
    separator="\t",
    infer_schema_length=1000,
    null_values=["", "null", "NULL", "None"]
)

print(gt.head())

print("\nSchema:")
print(gt.schema)

gt_cardinality = (
    gt.with_columns(
        pl.when(
            pl.col("matched_entity_ids").is_null()
            | (pl.col("matched_entity_ids").str.strip_chars() == "")
        )
        .then(0)
        .otherwise(
            pl.col("matched_entity_ids")
            .str.split(",")
            .list.len()
        )
        .alias("match_count")
    )
)

print("\nMatch-count distribution:")
print(
    gt_cardinality
    .group_by("match_count")
    .agg(pl.len().alias("s1_count"))
    .sort("match_count")
)

print("\nSingleton rate:")
singleton_rate = (
    gt_cardinality
    .select(
        (pl.col("match_count") == 0)
        .mean()
        .alias("singleton_fraction")
    )
    .item()
)

print(f"{singleton_rate:.4%}")

shape: (5, 2)
┌───────────────────┬─────────────────────────────────┐
│ source1_entity_id ┆ matched_entity_ids              │
│ ---               ┆ ---                             │
│ str               ┆ str                             │
╞═══════════════════╪═════════════════════════════════╡
│ S1-965667         ┆ S2-681193310,S2-743505751,S3-7… │
│ S1-55344266       ┆ S2-249013014,S2-197070651,S3-4… │
│ S1-343815751      ┆ S2-790675320,S2-479876582,S3-8… │
│ S1-656753428      ┆ S2-153058913,S2-24659151,S3-67… │
│ S1-102811957      ┆ S2-478959098,S2-553508714,S2-6… │
└───────────────────┴─────────────────────────────────┘

Schema:
Schema({'source1_entity_id': String, 'matched_entity_ids': String})

Match-count distribution:
shape: (12, 2)
┌─────────────┬──────────┐
│ match_count ┆ s1_count │
│ ---         ┆ ---      │
│ u32         ┆ u32      │
╞═════════════╪══════════╡
│ 0           ┆ 123247   │
│ 1           ┆ 119157   │
│ 2           ┆ 375212   │
│ 3           ┆ 530841   │
│ 4     

In [ ]:
# ============================================================
# AMLC 2026 — CELL 15
# Match composition: S2 vs S3
# ============================================================

def classify_match_set(s):

    if s is None or str(s).strip() == "":
        return "none"

    ids = [x.strip() for x in str(s).split(",") if x.strip()]

    has_s2 = any(x.startswith("S2-") for x in ids)
    has_s3 = any(x.startswith("S3-") for x in ids)

    if has_s2 and has_s3:
        return "both"
    elif has_s2:
        return "S2_only"
    elif has_s3:
        return "S3_only"
    else:
        return "other"


composition = (
    gt.with_columns(
        pl.col("matched_entity_ids")
        .map_elements(classify_match_set, return_dtype=pl.String)
        .alias("match_type")
    )
    .group_by("match_type")
    .agg(pl.len().alias("s1_count"))
    .sort("s1_count", descending=True)
)

print(composition)

shape: (4, 2)
┌────────────┬──────────┐
│ match_type ┆ s1_count │
│ ---        ┆ ---      │
│ str        ┆ u32      │
╞════════════╪══════════╡
│ both       ┆ 1776047  │
│ S3_only    ┆ 164498   │
│ S2_only    ┆ 143029   │
│ null       ┆ 123247   │
└────────────┴──────────┘


In [ ]:
# ============================================================
# AMLC 2026 — CELL 16
# Cross-entity exclusivity check
# ============================================================

edges = (
    gt
    .with_columns(
        pl.when(
            pl.col("matched_entity_ids").is_null()
            | (pl.col("matched_entity_ids").str.strip_chars() == "")
        )
        .then(pl.lit(None))
        .otherwise(
            pl.col("matched_entity_ids").str.split(",")
        )
        .alias("matched_list")
    )
    .explode("matched_list")
    .filter(
        pl.col("matched_list").is_not_null()
        & (pl.col("matched_list").str.strip_chars() != "")
    )
    .select([
        "source1_entity_id",
        pl.col("matched_list").alias("matched_entity_id")
    ])
)

print("Total positive edges:", edges.height)

uniqueness = (
    edges
    .group_by("matched_entity_id")
    .agg(
        pl.col("source1_entity_id").n_unique().alias("s1_count")
    )
)

print("\nHow many S2/S3 entities map to:")
print(
    uniqueness
    .group_by("s1_count")
    .agg(pl.len().alias("entity_count"))
    .sort("s1_count")
)

print("\nMaximum number of S1 entities sharing one S2/S3 ID:",
      uniqueness["s1_count"].max())

Total positive edges: 7638365

How many S2/S3 entities map to:
shape: (1, 2)
┌──────────┬──────────────┐
│ s1_count ┆ entity_count │
│ ---      ┆ ---          │
│ u32      ┆ u32          │
╞══════════╪══════════════╡
│ 1        ┆ 7638365      │
└──────────┴──────────────┘

Maximum number of S1 entities sharing one S2/S3 ID: 1


In [ ]:
# ============================================================
# AMLC 2026 — CELL 17
# Full match cardinality table
# ============================================================

print(
    gt_cardinality
    .group_by("match_count")
    .agg(pl.len().alias("s1_count"))
    .sort("match_count")
    .to_pandas()
    .to_string(index=False)
)

 match_count  s1_count
           0    123247
           1    119157
           2    375212
           3    530841
           4    484115
           5    321957
           6    164868
           7     63968
           8     18680
           9      4205
          10       534
          11        37


In [ ]:
# ============================================================
# AMLC 2026 — CELL 18
# Detailed text-length distributions
# ============================================================

for name in [
    "train_source1",
    "train_source2",
    "train_source3",
]:

    print("\n" + "=" * 90)
    print(name)
    print("=" * 90)

    lf = (
        scan_tsv(files[name])
        .select([
            pl.col("business_name")
              .str.len_chars()
              .alias("name_len"),

            pl.col("business_address")
              .fill_null("")
              .str.len_chars()
              .alias("address_len"),
        ])
    )

    result = (
        lf.select([
            pl.col("name_len").quantile(0.01).alias("name_p01"),
            pl.col("name_len").quantile(0.10).alias("name_p10"),
            pl.col("name_len").quantile(0.25).alias("name_p25"),
            pl.col("name_len").median().alias("name_p50"),
            pl.col("name_len").quantile(0.75).alias("name_p75"),
            pl.col("name_len").quantile(0.90).alias("name_p90"),
            pl.col("name_len").quantile(0.99).alias("name_p99"),

            pl.col("address_len").quantile(0.01).alias("addr_p01"),
            pl.col("address_len").quantile(0.10).alias("addr_p10"),
            pl.col("address_len").quantile(0.25).alias("addr_p25"),
            pl.col("address_len").median().alias("addr_p50"),
            pl.col("address_len").quantile(0.75).alias("addr_p75"),
            pl.col("address_len").quantile(0.90).alias("addr_p90"),
            pl.col("address_len").quantile(0.99).alias("addr_p99"),
        ])
        .collect()
    )

    print(result)


train_source1
shape: (1, 14)
┌──────────┬──────────┬──────────┬──────────┬───┬──────────┬──────────┬──────────┬──────────┐
│ name_p01 ┆ name_p10 ┆ name_p25 ┆ name_p50 ┆ … ┆ addr_p50 ┆ addr_p75 ┆ addr_p90 ┆ addr_p99 │
│ ---      ┆ ---      ┆ ---      ┆ ---      ┆   ┆ ---      ┆ ---      ┆ ---      ┆ ---      │
│ f64      ┆ f64      ┆ f64      ┆ f64      ┆   ┆ f64      ┆ f64      ┆ f64      ┆ f64      │
╞══════════╪══════════╪══════════╪══════════╪═══╪══════════╪══════════╪══════════╪══════════╡
│ 8.0      ┆ 14.0     ┆ 18.0     ┆ 24.0     ┆ … ┆ 41.0     ┆ 70.0     ┆ 90.0     ┆ 124.0    │
└──────────┴──────────┴──────────┴──────────┴───┴──────────┴──────────┴──────────┴──────────┘

train_source2
shape: (1, 14)
┌──────────┬──────────┬──────────┬──────────┬───┬──────────┬──────────┬──────────┬──────────┐
│ name_p01 ┆ name_p10 ┆ name_p25 ┆ name_p50 ┆ … ┆ addr_p50 ┆ addr_p75 ┆ addr_p90 ┆ addr_p99 │
│ ---      ┆ ---      ┆ ---      ┆ ---      ┆   ┆ ---      ┆ ---      ┆ ---      ┆ ---      │


In [ ]:
# ============================================================
# AMLC 2026 — CELL 19
# Unicode / script diagnostics
# ============================================================

import re

def script_flags(text):

    if text is None:
        text = ""

    text = str(text)

    return {
        "latin": int(bool(re.search(r"[A-Za-z]", text))),
        "devanagari": int(bool(re.search(r"[\u0900-\u097F]", text))),
        "tamil": int(bool(re.search(r"[\u0B80-\u0BFF]", text))),
        "kannada": int(bool(re.search(r"[\u0C80-\u0CFF]", text))),
        "telugu": int(bool(re.search(r"[\u0C00-\u0C7F]", text))),
        "malayalam": int(bool(re.search(r"[\u0D00-\u0D7F]", text))),
        "bengali": int(bool(re.search(r"[\u0980-\u09FF]", text))),
        "gujarati": int(bool(re.search(r"[\u0A80-\u0AFF]", text))),
        "gurmukhi": int(bool(re.search(r"[\u0A00-\u0A7F]", text))),
        "arabic": int(bool(re.search(r"[\u0600-\u06FF]", text))),
        "non_ascii": int(any(ord(c) > 127 for c in text)),
    }


# Sample-based first pass.
# We deliberately do NOT materialize all 12M records yet.
for name in [
    "train_source1",
    "train_source2",
    "train_source3",
]:
    sample = (
        scan_tsv(files[name])
        .select("business_name")
        .head(200_000)
        .collect()
        .to_series()
        .to_list()
    )

    counts = {
        key: 0
        for key in script_flags("").keys()
    }

    for value in sample:
        flags = script_flags(value)
        for key, v in flags.items():
            counts[key] += v

    print("\n", name)
    print("-" * 60)

    for key, value in counts.items():
        print(f"{key:15s}: {value / len(sample):8.2%}")


 train_source1
------------------------------------------------------------
latin          :  100.00%
devanagari     :    0.00%
tamil          :    0.00%
kannada        :    0.00%
telugu         :    0.00%
malayalam      :    0.00%
bengali        :    0.00%
gujarati       :    0.00%
gurmukhi       :    0.00%
arabic         :    0.00%
non_ascii      :    0.00%

 train_source2
------------------------------------------------------------
latin          :   91.11%
devanagari     :    5.27%
tamil          :    0.62%
kannada        :    0.73%
telugu         :    0.78%
malayalam      :    0.37%
bengali        :    0.62%
gujarati       :    0.60%
gurmukhi       :    0.13%
arabic         :    0.00%
non_ascii      :   15.07%

 train_source3
------------------------------------------------------------
latin          :   95.38%
devanagari     :    3.02%
tamil          :    0.38%
kannada        :    0.42%
telugu         :    0.46%
malayalam      :    0.19%
bengali        :    0.35%
gujarati       

In [ ]:
# ============================================================
# AMLC 2026 — CELL 20
# Name collision analysis
# ============================================================

import re
import unicodedata

LEGAL_SUFFIXES = {
    "limited", "ltd", "ltd.",
    "private", "pvt",
    "inc", "inc.",
    "corporation", "corp", "corp.",
    "company", "co", "co.",
    "llc", "llp",
    "limitedliabilitycompany",
    "private limited",
}

def simple_normalize_name(s):

    if s is None:
        return ""

    s = unicodedata.normalize("NFKC", str(s)).casefold()

    s = s.replace("&", " and ")

    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    tokens = [
        t for t in s.split()
        if t not in LEGAL_SUFFIXES
    ]

    return " ".join(tokens)


# IMPORTANT:
# Start with a 300k sample.
# We are measuring collision structure, not building the final index.
sample = (
    scan_tsv(files["train_source1"])
    .select([
        "entity_id",
        "business_name"
    ])
    .head(300_000)
    .collect()
)

sample = sample.with_columns(
    pl.col("business_name")
    .map_elements(
        simple_normalize_name,
        return_dtype=pl.String
    )
    .alias("name_norm")
)

collisions = (
    sample
    .group_by("name_norm")
    .agg([
        pl.len().alias("count"),
        pl.col("entity_id").n_unique().alias("unique_ids")
    ])
    .sort("count", descending=True)
)

print(collisions.head(50))

shape: (50, 3)
┌────────────────────────────┬───────┬────────────┐
│ name_norm                  ┆ count ┆ unique_ids │
│ ---                        ┆ ---   ┆ ---        │
│ str                        ┆ u32   ┆ u32        │
╞════════════════════════════╪═══════╪════════════╡
│ meridian                   ┆ 64    ┆ 64         │
│ redwood                    ┆ 49    ┆ 49         │
│ beacon                     ┆ 47    ┆ 47         │
│ aurora                     ┆ 47    ┆ 47         │
│ summit                     ┆ 46    ┆ 46         │
│ …                          ┆ …     ┆ …          │
│ amber                      ┆ 38    ┆ 38         │
│ chiropractic associates    ┆ 38    ┆ 38         │
│ internal medicine partners ┆ 38    ┆ 38         │
│ cascade                    ┆ 38    ┆ 38         │
│ basalt                     ┆ 38    ┆ 38         │
└────────────────────────────┴───────┴────────────┘


In [9]:
# ============================================================
# AMLC 2026 — CELL 21
# Unicode transliteration support
# ============================================================

!pip -q install anyascii

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 9.0 MB/s eta 0:00:00


In [ ]:
from anyascii import anyascii

print(anyascii("श्री साईं इन्फ्राटेक"))
print(anyascii("ஈஸ்டர்ன் கன்சல்டன்சி"))
print(anyascii("શ્રી સાઈ"))

sri saim inphratek
istrn kncltnci
sri sai


In [ ]:
# ============================================================
# AMLC 2026 — CELL 22
# Dual-view name normalization
# ============================================================

import re
import unicodedata
from anyascii import anyascii

LEGAL_SUFFIXES = {
    "limited",
    "ltd",
    "private",
    "pvt",
    "inc",
    "corporation",
    "corp",
    "company",
    "co",
    "llc",
    "llp",
    "sarl",
    "sasu",
    "sas",
    "plc",
    "gmbh",
}


def normalize_unicode(text):
    """
    Preserve native script.
    Used for exact Unicode / token comparisons.
    """
    if text is None:
        return ""

    text = unicodedata.normalize("NFKC", str(text)).casefold()
    text = text.replace("&", " and ")

    text = re.sub(
        r"[^\w\s]",
        " ",
        text,
        flags=re.UNICODE
    )

    text = re.sub(r"\s+", " ", text).strip()

    return text


def normalize_translit(text):
    """
    Transliterate to ASCII, then normalize.
    Used for cross-script matching.
    """
    if text is None:
        return ""

    text = anyascii(str(text))
    text = unicodedata.normalize("NFKC", text).casefold()
    text = text.replace("&", " and ")

    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def normalize_name_views(text):
    """
    Returns both representations.
    """
    native = normalize_unicode(text)
    translit = normalize_translit(text)

    return native, translit


examples = [
    "Orelee's Barbershop",
    "SHIVSHAKTI VIDYALAYA",
    "श्री साईं इन्फ्राटेक",
    "ஈஸ்டர்ன் கன்சல்டன்சி",
    "Fractales Amis Groupe S.A.S",
]

for x in examples:
    native, translit = normalize_name_views(x)

    print("\nRAW     :", x)
    print("NATIVE  :", native)
    print("TRANSLIT:", translit)


RAW     : Orelee's Barbershop
NATIVE  : orelee s barbershop
TRANSLIT: orelee s barbershop

RAW     : SHIVSHAKTI VIDYALAYA
NATIVE  : shivshakti vidyalaya
TRANSLIT: shivshakti vidyalaya

RAW     : श्री साईं इन्फ्राटेक
NATIVE  : श र स ई इन फ र ट क
TRANSLIT: sri saim inphratek

RAW     : ஈஸ்டர்ன் கன்சல்டன்சி
NATIVE  : ஈஸ டர ன கன சல டன ச
TRANSLIT: istrn kncltnci

RAW     : Fractales Amis Groupe S.A.S
NATIVE  : fractales amis groupe s a s
TRANSLIT: fractales amis groupe s a s


In [ ]:
# ============================================================
# AMLC 2026 — CELL 23
# Ground-truth positive edge table
# ============================================================

gt_edges = (
    gt
    .with_columns(
        pl.when(
            pl.col("matched_entity_ids").is_null()
            | (pl.col("matched_entity_ids").str.strip_chars() == "")
        )
        .then(pl.lit(None))
        .otherwise(
            pl.col("matched_entity_ids").str.split(",")
        )
        .alias("matched_list")
    )
    .explode("matched_list")
    .filter(
        pl.col("matched_list").is_not_null()
        & (pl.col("matched_list").str.strip_chars() != "")
    )
    .select([
        "source1_entity_id",
        pl.col("matched_list").alias("matched_entity_id"),
    ])
    .with_columns(
        pl.when(
            pl.col("matched_entity_id").str.starts_with("S2-")
        )
        .then(pl.lit("S2"))
        .otherwise(pl.lit("S3"))
        .alias("matched_source")
    )
)

print(gt_edges.head(20))
print("\nRows:", gt_edges.height)
print("\nSource counts:")
print(
    gt_edges
    .group_by("matched_source")
    .agg(pl.len().alias("edges"))
)

shape: (20, 3)
┌───────────────────┬───────────────────┬────────────────┐
│ source1_entity_id ┆ matched_entity_id ┆ matched_source │
│ ---               ┆ ---               ┆ ---            │
│ str               ┆ str               ┆ str            │
╞═══════════════════╪═══════════════════╪════════════════╡
│ S1-965667         ┆ S2-681193310      ┆ S2             │
│ S1-965667         ┆ S2-743505751      ┆ S2             │
│ S1-965667         ┆ S3-775321672      ┆ S3             │
│ S1-965667         ┆ S3-11291185       ┆ S3             │
│ S1-965667         ┆ S3-860443364      ┆ S3             │
│ …                 ┆ …                 ┆ …              │
│ S1-102811957      ┆ S2-478959098      ┆ S2             │
│ S1-102811957      ┆ S2-553508714      ┆ S2             │
│ S1-102811957      ┆ S2-625774905      ┆ S2             │
│ S1-102811957      ┆ S3-728090388      ┆ S3             │
│ S1-102811957      ┆ S3-928796641      ┆ S3             │
└───────────────────┴───────────────────┴

In [ ]:
# ============================================================
# AMLC 2026 — CELL 24
# Deterministic 100k S1 evaluation sample
# ============================================================

EVAL_N = 100_000
SEED = 2026

# Sample S1 IDs directly from the already-loaded ground truth.
eval_gt_ids = (
    gt
    .sample(
        n=EVAL_N,
        seed=SEED
    )
    .select("source1_entity_id")
)

eval_id_list = eval_gt_ids["source1_entity_id"].to_list()

print("Selected S1 evaluation entities:", len(eval_id_list))

# Pull the corresponding source-1 records.
s1_eval = (
    scan_tsv(files["train_source1"])
    .filter(
        pl.col("entity_id").is_in(eval_id_list)
    )
    .collect()
)

print("\nS1 evaluation shape:", s1_eval.shape)
print("\nSample:")
print(s1_eval.head(10))

Selected S1 evaluation entities: 100000

S1 evaluation shape: (100000, 4)

Sample:
shape: (10, 4)
┌──────────────┬─────────────────────────────────┬─────────────────────────────────┬─────────┐
│ entity_id    ┆ business_name                   ┆ business_address                ┆ country │
│ ---          ┆ ---                             ┆ ---                             ┆ ---     │
│ str          ┆ str                             ┆ str                             ┆ str     │
╞══════════════╪═════════════════════════════════╪═════════════════════════════════╪═════════╡
│ S1-913301865 ┆ Technologies Pratyaksh Finserv… ┆ Village Kurali, Sabhapur Road … ┆ India   │
│ S1-424529930 ┆ Griffin International Center    ┆ 536 College Avenue, City Of Wa… ┆ US      │
│ S1-846529766 ┆ Pocius, Herrera & Lima Connect… ┆ 4831 Beaumont Street, Simi Val… ┆ US      │
│ S1-8092452   ┆ Urban Nails!                    ┆ 1262 Grandstaff Avenue, Lancas… ┆ US      │
│ S1-831843558 ┆ Hernandez Regional Aktiengesel

In [ ]:
# ============================================================
# AMLC 2026 — CELL 25
# Evaluation subset ground truth
# ============================================================

eval_gt = (
    gt_edges
    .filter(
        pl.col("source1_entity_id")
        .is_in(eval_id_list)
    )
)

print("S1 eval entities :", s1_eval.height)
print("Positive edges   :", eval_gt.height)

print("\nPositive edges by source:")
print(
    eval_gt
    .group_by("matched_source")
    .agg(pl.len().alias("edges"))
    .sort("matched_source")
)

S1 eval entities : 100000
Positive edges   : 345980

Positive edges by source:
shape: (2, 2)
┌────────────────┬────────┐
│ matched_source ┆ edges  │
│ ---            ┆ ---    │
│ str            ┆ u32    │
╞════════════════╪════════╡
│ S2             ┆ 167274 │
│ S3             ┆ 178706 │
└────────────────┴────────┘


In [ ]:
# ============================================================
# AMLC 2026 — CELL 26
# Prepare source-2/source-3 name tables for blocking experiment
# ============================================================

s2_eval_pool = (
    scan_tsv(files["train_source2"])
    .select([
        "entity_id",
        "business_name",
        "country",
    ])
    .collect()
)

s3_eval_pool = (
    scan_tsv(files["train_source3"])
    .select([
        "entity_id",
        "business_name",
        "country",
    ])
    .collect()
)

print("S2:", s2_eval_pool.shape)
print("S3:", s3_eval_pool.shape)

print("\nRAM:")
!free -h

S2: (5034616, 3)
S3: (5285603, 3)

RAM:
               total        used        free      shared  buff/cache   available
Mem:            12Gi       2.2Gi       236Mi       3.5Mi       6.8Gi        10Gi
Swap:             0B          0B          0B


In [ ]:
# ============================================================
# AMLC 2026 — CELL 27
# Memory cleanup before large-scale indexing
# ============================================================

import gc

for name in ["s2_eval_pool", "s3_eval_pool"]:
    if name in globals():
        del globals()[name]

gc.collect()

print("Memory after cleanup:")
!free -h

Memory after cleanup:
               total        used        free      shared  buff/cache   available
Mem:            12Gi       2.2Gi       255Mi       3.5Mi       6.8Gi        10Gi
Swap:             0B          0B          0B


In [ ]:
# ============================================================
# AMLC 2026 — CELL 28
# Fast vectorized name normalization
#
# This is ONLY the first blocking oracle.
# It is NOT our final normalizer.
# ============================================================

import polars as pl

LEGAL_SUFFIX_PATTERN = (
    r"(?i)\b("
    r"private\s+limited|"
    r"private|"
    r"pvt\s+ltd|"
    r"pvt|"
    r"limited|"
    r"ltd|"
    r"corporation|"
    r"corp|"
    r"company|"
    r"co|"
    r"incorporated|"
    r"inc|"
    r"llc|"
    r"llp|"
    r"sarl|"
    r"sasu|"
    r"sas|"
    r"plc|"
    r"gmbh"
    r")\b"
)


def fast_name_expr(column="business_name"):
    """
    Vectorized normalization.

    Keeps Unicode letters/digits.
    Removes punctuation.
    Normalizes ampersand.
    Removes common legal suffixes.
    Collapses whitespace.

    We preserve Unicode because cross-script handling
    is a separate experiment.
    """

    return (
        pl.col(column)
        .fill_null("")
        .str.to_lowercase()
        .str.replace_all("&", " and ")
        .str.replace_all(r"[^\p{L}\p{N}\s]", " ")
        .str.replace_all(LEGAL_SUFFIX_PATTERN, " ")
        .str.replace_all(r"\s+", " ")
        .str.strip_chars()
    )


# Quick sanity check
test_names = pl.DataFrame({
    "business_name": [
        "ABC Private Limited",
        "ABC Pvt. Ltd.",
        "A&B Corporation",
        "Fractales Amis Groupe S.A.S",
        "श्री साईं इन्फ्राटेक",
    ]
})

print(
    test_names.with_columns(
        fast_name_expr().alias("name_norm")
    )
)

shape: (5, 2)
┌─────────────────────────────┬─────────────────────────────┐
│ business_name               ┆ name_norm                   │
│ ---                         ┆ ---                         │
│ str                         ┆ str                         │
╞═════════════════════════════╪═════════════════════════════╡
│ ABC Private Limited         ┆ abc                         │
│ ABC Pvt. Ltd.               ┆ abc                         │
│ A&B Corporation             ┆ a and b                     │
│ Fractales Amis Groupe S.A.S ┆ fractales amis groupe s a s │
│ श्री साईं इन्फ्राटेक             ┆ श र स ई इन फ र ट क          │
└─────────────────────────────┴─────────────────────────────┘


In [ ]:
# ============================================================
# AMLC 2026 — CELL 29
# Normalize evaluation S1 names
# ============================================================

s1_eval_norm = (
    s1_eval
    .select([
        "entity_id",
        "business_name",
        "country",
    ])
    .with_columns(
        fast_name_expr().alias("name_norm")
    )
)

print(s1_eval_norm.head(20))

print("\nEmpty normalized names:")
print(
    s1_eval_norm
    .filter(pl.col("name_norm") == "")
    .height
)

shape: (20, 4)
┌──────────────┬─────────────────────────────────┬─────────┬─────────────────────────────────┐
│ entity_id    ┆ business_name                   ┆ country ┆ name_norm                       │
│ ---          ┆ ---                             ┆ ---     ┆ ---                             │
│ str          ┆ str                             ┆ str     ┆ str                             │
╞══════════════╪═════════════════════════════════╪═════════╪═════════════════════════════════╡
│ S1-913301865 ┆ Technologies Pratyaksh Finserv… ┆ India   ┆ technologies pratyaksh finserv… │
│ S1-424529930 ┆ Griffin International Center    ┆ US      ┆ griffin international center    │
│ S1-846529766 ┆ Pocius, Herrera & Lima Connect… ┆ US      ┆ pocius herrera and lima connec… │
│ S1-8092452   ┆ Urban Nails!                    ┆ US      ┆ urban nails                     │
│ S1-831843558 ┆ Hernandez Regional Aktiengesel… ┆ US      ┆ hernandez regional aktiengesel… │
│ …            ┆ …                 

In [ ]:
# ============================================================
# AMLC 2026 — CELL 30
# Streaming S2 normalized-name index
# ============================================================

from pathlib import Path

BLOCKING_DIR = Path("/content/AMLC2026/cache/blocking")
BLOCKING_DIR.mkdir(parents=True, exist_ok=True)

S2_NAME_INDEX = BLOCKING_DIR / "train_s2_name_norm.parquet"

print("Building S2 normalized-name index...")
print("Output:", S2_NAME_INDEX)

(
    scan_tsv(files["train_source2"])
    .select([
        "entity_id",
        "country",
        "business_name",
    ])
    .with_columns(
        fast_name_expr().alias("name_norm")
    )
    .select([
        "entity_id",
        "country",
        "name_norm",
    ])
    .sink_parquet(
        S2_NAME_INDEX,
        compression="zstd",
        statistics=True,
    )
)

print("\n✅ S2 index written.")
print(f"Size: {S2_NAME_INDEX.stat().st_size / (1024**2):.1f} MB")

Building S2 normalized-name index...
Output: /content/AMLC2026/cache/blocking/train_s2_name_norm.parquet

✅ S2 index written.
Size: 77.2 MB


In [ ]:
# ============================================================
# AMLC 2026 — CELL 31
# Streaming S3 normalized-name index
# ============================================================

S3_NAME_INDEX = BLOCKING_DIR / "train_s3_name_norm.parquet"

print("Building S3 normalized-name index...")
print("Output:", S3_NAME_INDEX)

(
    scan_tsv(files["train_source3"])
    .select([
        "entity_id",
        "country",
        "business_name",
    ])
    .with_columns(
        fast_name_expr().alias("name_norm")
    )
    .select([
        "entity_id",
        "country",
        "name_norm",
    ])
    .sink_parquet(
        S3_NAME_INDEX,
        compression="zstd",
        statistics=True,
    )
)

print("\n✅ S3 index written.")
print(f"Size: {S3_NAME_INDEX.stat().st_size / (1024**2):.1f} MB")

Building S3 normalized-name index...
Output: /content/AMLC2026/cache/blocking/train_s3_name_norm.parquet

✅ S3 index written.
Size: 80.4 MB


In [ ]:
# ============================================================
# AMLC 2026 — CELL 32
# Verify normalized indexes
# ============================================================

print("S2 index:")
s2_idx = pl.scan_parquet(S2_NAME_INDEX)

print(s2_idx.collect_schema())

print(
    s2_idx
    .select(pl.len())
    .collect()
)

print("\nS3 index:")
s3_idx = pl.scan_parquet(S3_NAME_INDEX)

print(s3_idx.collect_schema())

print(
    s3_idx
    .select(pl.len())
    .collect()
)

print("\nDisk:")
!df -h /content

S2 index:
Schema({'entity_id': String, 'country': String, 'name_norm': String})
shape: (1, 1)
┌─────────┐
│ len     │
│ ---     │
│ u32     │
╞═════════╡
│ 5034616 │
└─────────┘

S3 index:
Schema({'entity_id': String, 'country': String, 'name_norm': String})
shape: (1, 1)
┌─────────┐
│ len     │
│ ---     │
│ u32     │
╞═════════╡
│ 5285603 │
└─────────┘

Disk:
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   53G   61G  47% /


In [ ]:
# ============================================================
# AMLC 2026 — CELL 33
# Exact normalized-name blocker recall
# ============================================================

# Convert the small evaluation-side tables to LazyFrames.
s1_norm_lookup_lf = s1_norm_lookup.lazy()
eval_gt_lf = eval_gt.lazy()

# -------------------------
# S2 positive edges
# -------------------------

s2_edge_eval = (
    eval_gt_lf
    .filter(pl.col("matched_source") == "S2")
    .join(
        s2_idx.select([
            pl.col("entity_id").alias("matched_entity_id"),
            pl.col("name_norm").alias("matched_name_norm"),
        ]),
        on="matched_entity_id",
        how="left",
    )
    .join(
        s1_norm_lookup_lf,
        on="source1_entity_id",
        how="left",
    )
    .with_columns(
        (
            (pl.col("name_norm") != "")
            & (pl.col("name_norm") == pl.col("matched_name_norm"))
        ).alias("name_hit")
    )
)

# -------------------------
# S3 positive edges
# -------------------------

s3_edge_eval = (
    eval_gt_lf
    .filter(pl.col("matched_source") == "S3")
    .join(
        s3_idx.select([
            pl.col("entity_id").alias("matched_entity_id"),
            pl.col("name_norm").alias("matched_name_norm"),
        ]),
        on="matched_entity_id",
        how="left",
    )
    .join(
        s1_norm_lookup_lf,
        on="source1_entity_id",
        how="left",
    )
    .with_columns(
        (
            (pl.col("name_norm") != "")
            & (pl.col("name_norm") == pl.col("matched_name_norm"))
        ).alias("name_hit")
    )
)

edge_eval = pl.concat([
    s2_edge_eval,
    s3_edge_eval,
]).collect()

print("=" * 80)
print("EDGE RECALL")
print("=" * 80)

print(
    edge_eval
    .group_by("matched_source")
    .agg([
        pl.len().alias("true_edges"),
        pl.col("name_hit").sum().alias("recovered_edges"),
        pl.col("name_hit").mean().alias("edge_recall"),
    ])
)

print("\nALL SOURCES")

print(
    edge_eval
    .select([
        pl.len().alias("true_edges"),
        pl.col("name_hit").sum().alias("recovered_edges"),
        pl.col("name_hit").mean().alias("edge_recall"),
    ])
)

EDGE RECALL
shape: (2, 4)
┌────────────────┬────────────┬─────────────────┬─────────────┐
│ matched_source ┆ true_edges ┆ recovered_edges ┆ edge_recall │
│ ---            ┆ ---        ┆ ---             ┆ ---         │
│ str            ┆ u32        ┆ u32             ┆ f64         │
╞════════════════╪════════════╪═════════════════╪═════════════╡
│ S2             ┆ 167274     ┆ 71337           ┆ 0.426468    │
│ S3             ┆ 178706     ┆ 73339           ┆ 0.410389    │
└────────────────┴────────────┴─────────────────┴─────────────┘

ALL SOURCES
shape: (1, 3)
┌────────────┬─────────────────┬─────────────┐
│ true_edges ┆ recovered_edges ┆ edge_recall │
│ ---        ┆ ---             ┆ ---         │
│ u32        ┆ u32             ┆ f64         │
╞════════════╪═════════════════╪═════════════╡
│ 345980     ┆ 144676          ┆ 0.418163    │
└────────────┴─────────────────┴─────────────┘


In [ ]:
# ============================================================
# AMLC 2026 — CELL 34
# Per-S1 full candidate recall
# ============================================================

per_s1_recall = (
    edge_eval
    .group_by("source1_entity_id")
    .agg([
        pl.len().alias("true_match_count"),
        pl.col("name_hit").sum().alias("recovered_match_count"),
    ])
    .with_columns(
        (
            pl.col("recovered_match_count")
            == pl.col("true_match_count")
        ).alias("fully_recovered")
    )
)

print("=" * 80)
print("S1 FULL-SET RECALL")
print("=" * 80)

print(
    per_s1_recall
    .select([
        pl.len().alias("evaluated_s1"),
        pl.col("fully_recovered").sum().alias("fully_recovered_s1"),
        pl.col("fully_recovered").mean().alias("full_recall"),
    ])
)

print("\nFull recall by true cardinality:")

print(
    per_s1_recall
    .group_by("true_match_count")
    .agg([
        pl.len().alias("s1_count"),
        pl.col("fully_recovered").mean().alias("full_recall"),
    ])
    .sort("true_match_count")
)

S1 FULL-SET RECALL
shape: (1, 3)
┌──────────────┬────────────────────┬─────────────┐
│ evaluated_s1 ┆ fully_recovered_s1 ┆ full_recall │
│ ---          ┆ ---                ┆ ---         │
│ u32          ┆ u32                ┆ f64         │
╞══════════════╪════════════════════╪═════════════╡
│ 94404        ┆ 9054               ┆ 0.095907    │
└──────────────┴────────────────────┴─────────────┘

Full recall by true cardinality:
shape: (11, 3)
┌──────────────────┬──────────┬─────────────┐
│ true_match_count ┆ s1_count ┆ full_recall │
│ ---              ┆ ---      ┆ ---         │
│ u32              ┆ u32      ┆ f64         │
╞══════════════════╪══════════╪═════════════╡
│ 1                ┆ 5398     ┆ 0.420897    │
│ 2                ┆ 17069    ┆ 0.189818    │
│ 3                ┆ 24058    ┆ 0.089617    │
│ 4                ┆ 21786    ┆ 0.044249    │
│ 5                ┆ 14668    ┆ 0.022293    │
│ …                ┆ …        ┆ …           │
│ 7                ┆ 2876     ┆ 0.005911    │
│ 

In [ ]:
# ============================================================
# AMLC 2026 — CELL 35
# Candidate-volume statistics for exact normalized-name blocking
# ============================================================

s1_norm_for_counts_lf = s1_norm_lookup.lazy()


def candidate_count_by_name_lazy(
    s1_norm_df_lf,
    source_index_lf,
    prefix,
):
    counts = (
        source_index_lf
        .filter(pl.col("name_norm") != "")
        .group_by("name_norm")
        .agg(
            pl.len().alias(f"{prefix}_candidate_count")
        )
    )

    return (
        s1_norm_df_lf
        .select([
            "source1_entity_id",
            "name_norm",
        ])
        .join(
            counts,
            on="name_norm",
            how="left",
        )
        .with_columns(
            pl.col(f"{prefix}_candidate_count")
            .fill_null(0)
        )
    )


s2_counts = candidate_count_by_name_lazy(
    s1_norm_for_counts_lf,
    s2_idx,
    "s2",
)

s3_counts = candidate_count_by_name_lazy(
    s1_norm_for_counts_lf,
    s3_idx,
    "s3",
)

candidate_counts = (
    s2_counts
    .join(
        s3_counts.select([
            "source1_entity_id",
            "s3_candidate_count",
        ]),
        on="source1_entity_id",
    )
    .with_columns(
        (
            pl.col("s2_candidate_count")
            + pl.col("s3_candidate_count")
        ).alias("total_candidate_count")
    )
)

print("=" * 80)
print("EXACT NORMALIZED-NAME CANDIDATE VOLUME")
print("=" * 80)

result = (
    candidate_counts
    .select([
        pl.col("total_candidate_count")
            .mean()
            .alias("mean"),

        pl.col("total_candidate_count")
            .median()
            .alias("median"),

        pl.col("total_candidate_count")
            .quantile(0.90)
            .alias("p90"),

        pl.col("total_candidate_count")
            .quantile(0.95)
            .alias("p95"),

        pl.col("total_candidate_count")
            .quantile(0.99)
            .alias("p99"),

        pl.col("total_candidate_count")
            .max()
            .alias("max"),

        (
            pl.col("total_candidate_count") == 0
        )
        .mean()
        .alias("zero_candidate_fraction"),
    ])
    .collect()
)

print(result)

EXACT NORMALIZED-NAME CANDIDATE VOLUME
shape: (1, 7)
┌──────────┬────────┬──────┬───────┬───────┬──────┬─────────────────────────┐
│ mean     ┆ median ┆ p90  ┆ p95   ┆ p99   ┆ max  ┆ zero_candidate_fraction │
│ ---      ┆ ---    ┆ ---  ┆ ---   ┆ ---   ┆ ---  ┆ ---                     │
│ f64      ┆ f64    ┆ f64  ┆ f64   ┆ f64   ┆ u32  ┆ f64                     │
╞══════════╪════════╪══════╪═══════╪═══════╪══════╪═════════════════════════╡
│ 35.38077 ┆ 3.0    ┆ 73.0 ┆ 153.0 ┆ 601.0 ┆ 1348 ┆ 0.1048                  │
└──────────┴────────┴──────┴───────┴───────┴──────┴─────────────────────────┘


In [ ]:
# ============================================================
# AMLC 2026 — CELL 36
# Inspect missed positive matches
# ============================================================

MISSED_SAMPLE_N = 1000

missed_edges = (
    edge_eval
    .filter(
        ~pl.col("name_hit")
    )
    .sample(
        n=min(MISSED_SAMPLE_N, edge_eval.filter(~pl.col("name_hit")).height),
        seed=2026,
    )
)

print("Missed positive edges sampled:", missed_edges.height)
print(missed_edges.head(20))

Missed positive edges sampled: 1000
shape: (20, 6)
┌─────────────────┬─────────────────┬────────────────┬─────────────────┬────────────────┬──────────┐
│ source1_entity_ ┆ matched_entity_ ┆ matched_source ┆ matched_name_no ┆ name_norm      ┆ name_hit │
│ id              ┆ id              ┆ ---            ┆ rm              ┆ ---            ┆ ---      │
│ ---             ┆ ---             ┆ str            ┆ ---             ┆ str            ┆ bool     │
│ str             ┆ str             ┆                ┆ str             ┆                ┆          │
╞═════════════════╪═════════════════╪════════════════╪═════════════════╪════════════════╪══════════╡
│ S1-762526497    ┆ S2-991263471    ┆ S2             ┆ aparna sérvices ┆ aparna academy ┆ false    │
│ S1-835687699    ┆ S3-744776519    ┆ S3             ┆ atlantic guild  ┆ atlantic guild ┆ false    │
│                 ┆                 ┆                ┆ board           ┆                ┆          │
│ S1-673307798    ┆ S3-376720015    ┆ S3

In [ ]:
# ============================================================
# AMLC 2026 — CELL 37
# Pull S1 + matched source records for missed edges
# ============================================================

miss_ids = missed_edges["source1_entity_id"].to_list()
matched_ids = missed_edges["matched_entity_id"].to_list()

# S1 records
s1_missed = (
    scan_tsv(files["train_source1"])
    .filter(
        pl.col("entity_id").is_in(miss_ids)
    )
    .select([
        pl.col("entity_id").alias("source1_entity_id"),
        pl.col("business_name").alias("s1_name"),
        pl.col("business_address").alias("s1_address"),
        pl.col("country").alias("s1_country"),
    ])
    .collect()
)

# S2 records
s2_missed = (
    scan_tsv(files["train_source2"])
    .filter(
        pl.col("entity_id").is_in(matched_ids)
    )
    .select([
        pl.col("entity_id").alias("matched_entity_id"),
        pl.col("business_name").alias("matched_name"),
        pl.col("business_address").alias("matched_address"),
        pl.col("country").alias("matched_country"),
    ])
    .collect()
)

# S3 records
s3_missed = (
    scan_tsv(files["train_source3"])
    .filter(
        pl.col("entity_id").is_in(matched_ids)
    )
    .select([
        pl.col("entity_id").alias("matched_entity_id"),
        pl.col("business_name").alias("matched_name"),
        pl.col("business_address").alias("matched_address"),
        pl.col("country").alias("matched_country"),
    ])
    .collect()
)

source_records = pl.concat([
    s2_missed,
    s3_missed,
])

missed_inspection = (
    missed_edges
    .join(
        s1_missed,
        on="source1_entity_id",
        how="left",
    )
    .join(
        source_records,
        on="matched_entity_id",
        how="left",
    )
)

print(
    missed_inspection
    .select([
        "source1_entity_id",
        "matched_entity_id",
        "matched_source",
        "s1_country",
        "matched_country",
        "s1_name",
        "matched_name",
        "s1_address",
        "matched_address",
    ])
    .head(50)
)

shape: (50, 9)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ source1_e ┆ matched_e ┆ matched_s ┆ s1_countr ┆ … ┆ s1_name   ┆ matched_n ┆ s1_addres ┆ matched_ │
│ ntity_id  ┆ ntity_id  ┆ ource     ┆ y         ┆   ┆ ---       ┆ ame       ┆ s         ┆ address  │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ str       ┆ ---       ┆ ---       ┆ ---      │
│ str       ┆ str       ┆ str       ┆ str       ┆   ┆           ┆ str       ┆ str       ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ S1-762526 ┆ S2-991263 ┆ S2        ┆ India     ┆ … ┆ Aparna    ┆ Aparna    ┆ Bungalow  ┆ ગુજરાત,   │
│ 497       ┆ 471       ┆           ┆           ┆   ┆ Academy   ┆ (Sérvices ┆ No.10,    ┆ BUNGALOW │
│           ┆           ┆           ┆           ┆   ┆           ┆ )         ┆ Vraj Gopi ┆ NO.10,   │
│           ┆           ┆           ┆           ┆   ┆           ┆          

In [ ]:
# ============================================================
# AMLC 2026 — CELL 38
# Country consistency on true matches
# ============================================================

# Get all S1 countries for the sampled evaluation IDs.
s1_country_eval = (
    scan_tsv(files["train_source1"])
    .filter(
        pl.col("entity_id").is_in(eval_id_list)
    )
    .select([
        pl.col("entity_id").alias("source1_entity_id"),
        pl.col("country").alias("s1_country"),
    ])
    .collect()
)

# Get country of matched S2/S3 IDs.
s2_country = (
    scan_tsv(files["train_source2"])
    .filter(
        pl.col("entity_id").is_in(
            eval_gt
            .filter(pl.col("matched_source") == "S2")
            ["matched_entity_id"]
            .to_list()
        )
    )
    .select([
        pl.col("entity_id").alias("matched_entity_id"),
        pl.col("country").alias("matched_country"),
    ])
    .collect()
)

s3_country = (
    scan_tsv(files["train_source3"])
    .filter(
        pl.col("entity_id").is_in(
            eval_gt
            .filter(pl.col("matched_source") == "S3")
            ["matched_entity_id"]
            .to_list()
        )
    )
    .select([
        pl.col("entity_id").alias("matched_entity_id"),
        pl.col("country").alias("matched_country"),
    ])
    .collect()
)

country_eval = (
    eval_gt
    .join(
        s1_country_eval,
        on="source1_entity_id",
        how="left",
    )
    .join(
        pl.concat([s2_country, s3_country]),
        on="matched_entity_id",
        how="left",
    )
    .with_columns(
        (
            pl.col("s1_country")
            == pl.col("matched_country")
        ).alias("same_country")
    )
)

print(
    country_eval
    .group_by("matched_source")
    .agg([
        pl.len().alias("true_edges"),
        pl.col("same_country").sum().alias("same_country_edges"),
        pl.col("same_country").mean().alias("country_consistency"),
    ])
)

print("\nOverall:")
print(
    country_eval
    .select([
        pl.len().alias("true_edges"),
        pl.col("same_country").mean().alias("country_consistency"),
    ])
)

shape: (2, 4)
┌────────────────┬────────────┬────────────────────┬─────────────────────┐
│ matched_source ┆ true_edges ┆ same_country_edges ┆ country_consistency │
│ ---            ┆ ---        ┆ ---                ┆ ---                 │
│ str            ┆ u32        ┆ u32                ┆ f64                 │
╞════════════════╪════════════╪════════════════════╪═════════════════════╡
│ S3             ┆ 178706     ┆ 178706             ┆ 1.0                 │
│ S2             ┆ 167274     ┆ 167274             ┆ 1.0                 │
└────────────────┴────────────┴────────────────────┴─────────────────────┘

Overall:
shape: (1, 2)
┌────────────┬─────────────────────┐
│ true_edges ┆ country_consistency │
│ ---        ┆ ---                 │
│ u32        ┆ f64                 │
╞════════════╪═════════════════════╡
│ 345980     ┆ 1.0                 │
└────────────┴─────────────────────┘


In [ ]:
# ============================================================
# AMLC 2026 — CELL 39
# PERSISTENT CHECKPOINT SETUP
# ============================================================

from pathlib import Path

DRIVE_PROJECT = Path("/content/drive/MyDrive/AMLC2026")

DRIVE_CACHE = DRIVE_PROJECT / "cache"
DRIVE_BLOCKING = DRIVE_CACHE / "blocking"
DRIVE_STATE = DRIVE_CACHE / "state"
DRIVE_ER_EDA = DRIVE_STATE / "er_eda"

for p in [
    DRIVE_CACHE,
    DRIVE_BLOCKING,
    DRIVE_STATE,
    DRIVE_ER_EDA,
]:
    p.mkdir(parents=True, exist_ok=True)

print("Persistent directories ready:")
print("PROJECT :", DRIVE_PROJECT)
print("CACHE   :", DRIVE_CACHE)
print("BLOCKING:", DRIVE_BLOCKING)
print("STATE   :", DRIVE_STATE)
print("ER EDA  :", DRIVE_ER_EDA)

Persistent directories ready:
PROJECT : /content/drive/MyDrive/AMLC2026
CACHE   : /content/drive/MyDrive/AMLC2026/cache
BLOCKING: /content/drive/MyDrive/AMLC2026/cache/blocking
STATE   : /content/drive/MyDrive/AMLC2026/cache/state
ER EDA  : /content/drive/MyDrive/AMLC2026/cache/state/er_eda


In [ ]:
# ============================================================
# AMLC 2026 — CELL 40
# BACKUP NORMALIZED NAME INDEXES TO GOOGLE DRIVE
# ============================================================

import subprocess

LOCAL_BLOCKING = Path("/content/AMLC2026/cache/blocking")

print("Syncing blocking indexes to Google Drive...\n")

result = subprocess.run(
    [
        "rsync",
        "-ah",
        "--info=progress2",
        str(LOCAL_BLOCKING) + "/",
        str(DRIVE_BLOCKING) + "/",
    ],
    check=True,
)

print("\n✅ Blocking indexes backed up.")

for p in sorted(DRIVE_BLOCKING.glob("*")):
    print(
        f"{p.name:45s}"
        f"{p.stat().st_size / (1024**2):9.1f} MB"
    )

Syncing blocking indexes to Google Drive...


✅ Blocking indexes backed up.
train_s2_name_norm.parquet                        77.2 MB
train_s3_name_norm.parquet                        80.4 MB


In [ ]:
# ============================================================
# AMLC 2026 — CELL 41
# SAVE CURRENT EDA / VALIDATION STATE
# LazyFrame-safe version
# ============================================================

from pathlib import Path
import polars as pl

DRIVE_PROJECT = Path("/content/drive/MyDrive/AMLC2026")

DRIVE_ER_EDA = (
    DRIVE_PROJECT
    / "cache"
    / "state"
    / "er_eda"
)

DRIVE_ER_EDA.mkdir(parents=True, exist_ok=True)


def save_polars(obj, path):
    """
    Save either a Polars DataFrame or LazyFrame.
    LazyFrames are collected at checkpoint time.
    """

    if isinstance(obj, pl.LazyFrame):
        print(f"Collecting LazyFrame for checkpoint: {path.name}")
        obj = obj.collect()

    elif not isinstance(obj, pl.DataFrame):
        raise TypeError(
            f"Unsupported object type for {path.name}: "
            f"{type(obj)}"
        )

    obj.write_parquet(
        path,
        compression="zstd"
    )

    print(
        f"✅ {path.name:32s}"
        f"{path.stat().st_size / (1024**2):9.2f} MB"
    )


artifacts = {
    "gt.parquet": gt,
    "gt_edges.parquet": gt_edges,
    "s1_eval.parquet": s1_eval,
    "eval_gt.parquet": eval_gt,
    "edge_eval.parquet": edge_eval,
    "per_s1_recall.parquet": per_s1_recall,
    "candidate_counts.parquet": candidate_counts,
    "country_eval.parquet": country_eval,
    "missed_edges.parquet": missed_edges,
    "missed_inspection.parquet": missed_inspection,
}


print("=" * 80)
print("SAVING AMLC 2026 EDA CHECKPOINT")
print("=" * 80)

for filename, obj in artifacts.items():

    save_polars(
        obj,
        DRIVE_ER_EDA / filename
    )

print("\n✅ ALL EDA ARTIFACTS SAVED.")

SAVING AMLC 2026 EDA CHECKPOINT
✅ gt.parquet                          55.01 MB
✅ gt_edges.parquet                    66.81 MB
✅ s1_eval.parquet                      3.57 MB
✅ eval_gt.parquet                      3.02 MB
✅ edge_eval.parquet                    7.19 MB
✅ per_s1_recall.parquet                0.58 MB
✅ candidate_counts.parquet             1.55 MB
✅ country_eval.parquet                 3.08 MB
✅ missed_edges.parquet                 0.04 MB
✅ missed_inspection.parquet            0.12 MB

✅ ALL EDA ARTIFACTS SAVED.


In [ ]:
# ============================================================
# AMLC 2026 — CELL 42
# SAVE EXPERIMENT SUMMARY / FINDINGS
# ============================================================

import json

summary = {
    "checkpoint": "ER_EDA_001",
    "timestamp_note": "AMLC 2026 live run",

    "dataset": {
        "train_source1": 2206821,
        "train_source2": 5034616,
        "train_source3": 5285603,
        "train_ground_truth": 2206821,
        "test_source1": 1732544,
        "test_source2": 4887273,
        "test_source3": 5082316,
    },

    "ground_truth": {
        "positive_edges": 7638365,
        "singleton_count": 123247,
        "singleton_fraction": 0.055848,
        "mean_matches_per_s1": 7638365 / 2206821,
        "s2_positive_edges": 3693619,
        "s3_positive_edges": 3944746,
        "max_matches_per_s1": 11,
    },

    "match_cardinality": {
        "0": 123247,
        "1": 119157,
        "2": 375212,
        "3": 530841,
        "4": 484115,
        "5": 321957,
        "6": 164868,
        "7": 63968,
        "8": 18680,
        "9": 4205,
        "10": 534,
        "11": 37,
    },

    "composition": {
        "both_s2_s3": 1776047,
        "s2_only": 143029,
        "s3_only": 164498,
        "none": 123247,
    },

    "training_exclusivity_observed": {
        "positive_s2_s3_records_with_multiple_s1_assignments": 0,
        "maximum_s1_count_per_positive_s2_s3_record": 1,
    },

    "name_blocking_100k_eval": {
        "eval_s1": 100000,
        "positive_edges": 345980,
        "s2_edge_recall": 0.426468,
        "s3_edge_recall": 0.410389,
        "overall_edge_recall": 0.418163,
        "full_s1_recovery": 0.095907,
    },

    "country_consistency_100k_eval": {
        "overall": 1.0,
        "s2": 1.0,
        "s3": 1.0,
        "note": "Empirical training invariant; not stated as an explicit official rule."
    },

    "engineering": {
        "s2_normalized_index": "train_s2_name_norm.parquet",
        "s3_normalized_index": "train_s3_name_norm.parquet",
        "index_type": "Parquet / ZSTD",
    },

    "next_stage": [
        "Exact normalized-name candidate volume analysis",
        "Missed-positive error taxonomy",
        "Country hard-block validation on broader sample",
        "Character n-gram retrieval",
        "Token / rare-token blocking",
        "Address / numeric blocking",
        "Cross-script transliteration",
        "Semantic ANN only after lexical blockers are measured",
    ]
}

summary_path = DRIVE_ER_EDA / "checkpoint_summary.json"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("✅ Summary saved:")
print(summary_path)

✅ Summary saved:
/content/drive/MyDrive/AMLC2026/cache/state/er_eda/checkpoint_summary.json


In [ ]:
# ============================================================
# AMLC 2026 — CELL 43
# SAVE ENVIRONMENT MANIFEST
# ============================================================

import sys
import subprocess
import json
from pathlib import Path

manifest = {
    "python": sys.version,
    "packages": {}
}

packages_to_record = [
    "polars",
    "pandas",
    "numpy",
    "duckdb",
    "rapidfuzz",
    "scikit-learn",
    "lightgbm",
    "xgboost",
    "faiss-cpu",
    "anyascii",
]

for package in packages_to_record:
    try:
        output = subprocess.check_output(
            [sys.executable, "-m", "pip", "show", package],
            text=True
        )

        version = None

        for line in output.splitlines():
            if line.startswith("Version:"):
                version = line.split(":", 1)[1].strip()
                break

        manifest["packages"][package] = version

    except Exception:
        manifest["packages"][package] = None

manifest_path = DRIVE_PROJECT / "cache" / "environment_manifest.json"

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print(json.dumps(manifest, indent=2))
print("\n✅ Environment manifest saved.")

{
  "python": "3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]",
  "packages": {
    "polars": "1.35.2",
    "pandas": "2.2.3",
    "numpy": "2.1.3",
    "duckdb": "1.3.2",
    "rapidfuzz": "3.14.6",
    "scikit-learn": "1.6.1",
    "lightgbm": "4.6.0",
    "xgboost": "3.4.1",
    "faiss-cpu": "1.15.1",
    "anyascii": "0.3.3"
  }
}

✅ Environment manifest saved.


In [ ]:
# ============================================================
# AMLC 2026 — CELL 44
# 🚨 AMLC 2026 WAR-ROOM RESUME CELL
# Robust checkpoint restore
# ============================================================

from pathlib import Path
import json
import polars as pl

DRIVE_PROJECT = Path("/content/drive/MyDrive/AMLC2026")

DRIVE_BLOCKING = (
    DRIVE_PROJECT
    / "cache"
    / "blocking"
)

DRIVE_ER_EDA = (
    DRIVE_PROJECT
    / "cache"
    / "state"
    / "er_eda"
)

TRAIN_DIR = DRIVE_PROJECT / "dataset" / "train"
TEST_DIR = DRIVE_PROJECT / "dataset" / "test"

files = {
    "train_source1": TRAIN_DIR / "train_source1.tsv",
    "train_source2": TRAIN_DIR / "train_source2.tsv",
    "train_source3": TRAIN_DIR / "train_source3.tsv",
    "train_ground_truth": TRAIN_DIR / "train_ground_truth.tsv",
    "test_source1": TEST_DIR / "test_source1.tsv",
    "test_source2": TEST_DIR / "test_source2.tsv",
    "test_source3": TEST_DIR / "test_source3.tsv",
}


# ------------------------------------------------------------
# Loader
# ------------------------------------------------------------

def clean_column_name(name: str) -> str:
    return str(name).replace("\ufeff", "").strip()


def scan_tsv(path: Path) -> pl.LazyFrame:

    lf = pl.scan_csv(
        path,
        separator="\t",
        infer_schema_length=1000,
        null_values=["", "null", "NULL", "None"],
    )

    schema = lf.collect_schema()

    rename_map = {
        col: clean_column_name(col)
        for col in schema.names()
        if clean_column_name(col) != col
    }

    if rename_map:
        lf = lf.rename(rename_map)

    return lf


# ------------------------------------------------------------
# Normalize indexes
# ------------------------------------------------------------

S2_NAME_INDEX = DRIVE_BLOCKING / "train_s2_name_norm.parquet"
S3_NAME_INDEX = DRIVE_BLOCKING / "train_s3_name_norm.parquet"

assert S2_NAME_INDEX.exists(), f"Missing: {S2_NAME_INDEX}"
assert S3_NAME_INDEX.exists(), f"Missing: {S3_NAME_INDEX}"

s2_idx = pl.scan_parquet(S2_NAME_INDEX)
s3_idx = pl.scan_parquet(S3_NAME_INDEX)


# ------------------------------------------------------------
# Restore saved artifacts
# ------------------------------------------------------------

artifact_names = [
    "gt",
    "gt_edges",
    "s1_eval",
    "eval_gt",
    "edge_eval",
    "per_s1_recall",
    "candidate_counts",
    "country_eval",
    "missed_edges",
    "missed_inspection",
]

restored = []
missing = []

for name in artifact_names:

    path = DRIVE_ER_EDA / f"{name}.parquet"

    if path.exists():

        globals()[name] = pl.read_parquet(path)
        restored.append(name)

    else:

        missing.append(name)


# ------------------------------------------------------------
# Restore summary
# ------------------------------------------------------------

summary_path = DRIVE_ER_EDA / "checkpoint_summary.json"

if summary_path.exists():

    with open(summary_path, "r", encoding="utf-8") as f:
        checkpoint_summary = json.load(f)

else:

    checkpoint_summary = None


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("=" * 80)
print("AMLC 2026 WAR-ROOM CHECKPOINT")
print("=" * 80)

print("\n✅ Restored artifacts:")

for name in restored:
    obj = globals()[name]
    print(f"   {name:25s} {obj.shape}")

if missing:

    print("\n⚠️ Missing optional artifacts:")

    for name in missing:
        print(f"   {name}")

else:

    print("\n✅ No checkpoint artifacts missing.")

print("\nIndexes:")
print("   S2:", S2_NAME_INDEX)
print("   S3:", S3_NAME_INDEX)

if checkpoint_summary:

    print("\nKey findings:")

    print(
        "   Name edge recall:",
        checkpoint_summary[
            "name_blocking_100k_eval"
        ]["overall_edge_recall"]
    )

    print(
        "   Full S1 recall:",
        checkpoint_summary[
            "name_blocking_100k_eval"
        ]["full_s1_recovery"]
    )

    print(
        "   Country consistency:",
        checkpoint_summary[
            "country_consistency_100k_eval"
        ]["overall"]
    )

print("\n✅ RESUME STATE READY.")

AMLC 2026 WAR-ROOM CHECKPOINT

✅ Restored artifacts:
   gt                        (2206821, 2)
   gt_edges                  (7638365, 3)
   s1_eval                   (100000, 4)
   eval_gt                   (345980, 3)
   edge_eval                 (345980, 6)
   per_s1_recall             (94404, 4)
   candidate_counts          (100000, 5)
   country_eval              (345980, 6)
   missed_edges              (1000, 6)
   missed_inspection         (1000, 12)

✅ No checkpoint artifacts missing.

Indexes:
   S2: /content/drive/MyDrive/AMLC2026/cache/blocking/train_s2_name_norm.parquet
   S3: /content/drive/MyDrive/AMLC2026/cache/blocking/train_s3_name_norm.parquet

Key findings:
   Name edge recall: 0.418163
   Full S1 recall: 0.095907
   Country consistency: 1.0

✅ RESUME STATE READY.


In [ ]:
# ============================================================
# AMLC 2026 — CELL 45
# FINAL CHECKPOINT INTEGRITY TEST
# ============================================================

required_files = [
    DRIVE_BLOCKING / "train_s2_name_norm.parquet",
    DRIVE_BLOCKING / "train_s3_name_norm.parquet",

    DRIVE_ER_EDA / "gt.parquet",
    DRIVE_ER_EDA / "gt_edges.parquet",
    DRIVE_ER_EDA / "s1_eval.parquet",
    DRIVE_ER_EDA / "eval_gt.parquet",
    DRIVE_ER_EDA / "edge_eval.parquet",
    DRIVE_ER_EDA / "per_s1_recall.parquet",
    DRIVE_ER_EDA / "candidate_counts.parquet",
    DRIVE_ER_EDA / "country_eval.parquet",
    DRIVE_ER_EDA / "missed_edges.parquet",
    DRIVE_ER_EDA / "missed_inspection.parquet",
    DRIVE_ER_EDA / "checkpoint_summary.json",

    DRIVE_PROJECT / "cache" / "environment_manifest.json",
]

print("=" * 80)
print("CHECKPOINT INTEGRITY")
print("=" * 80)

all_ok = True

for path in required_files:

    exists = path.exists()

    print(
        ("✅" if exists else "❌"),
        path
    )

    if not exists:
        all_ok = False

print("\n" + "=" * 80)

if all_ok:
    print("✅ ALL CHECKPOINT ARTIFACTS ARE SAFE ON GOOGLE DRIVE.")
    print("✅ SAFE TO TAKE A BREAK / DISCONNECT.")
else:
    print("❌ CHECKPOINT INCOMPLETE — DO NOT DISCONNECT YET.")

print("=" * 80)

CHECKPOINT INTEGRITY
✅ /content/drive/MyDrive/AMLC2026/cache/blocking/train_s2_name_norm.parquet
✅ /content/drive/MyDrive/AMLC2026/cache/blocking/train_s3_name_norm.parquet
✅ /content/drive/MyDrive/AMLC2026/cache/state/er_eda/gt.parquet
✅ /content/drive/MyDrive/AMLC2026/cache/state/er_eda/gt_edges.parquet
✅ /content/drive/MyDrive/AMLC2026/cache/state/er_eda/s1_eval.parquet
✅ /content/drive/MyDrive/AMLC2026/cache/state/er_eda/eval_gt.parquet
✅ /content/drive/MyDrive/AMLC2026/cache/state/er_eda/edge_eval.parquet
✅ /content/drive/MyDrive/AMLC2026/cache/state/er_eda/per_s1_recall.parquet
✅ /content/drive/MyDrive/AMLC2026/cache/state/er_eda/candidate_counts.parquet
✅ /content/drive/MyDrive/AMLC2026/cache/state/er_eda/country_eval.parquet
✅ /content/drive/MyDrive/AMLC2026/cache/state/er_eda/missed_edges.parquet
✅ /content/drive/MyDrive/AMLC2026/cache/state/er_eda/missed_inspection.parquet
✅ /content/drive/MyDrive/AMLC2026/cache/state/er_eda/checkpoint_summary.json
✅ /content/drive/MyDrive/AM

In [6]:
# ============================================================
# AMLC 2026 — CELL 46
# WAR-ROOM RESUME / NEXT-STAGE STATE
#
# Purpose:
#   Reconstruct only the small state needed for the next stage.
#   Do NOT reload 12.5M source rows into RAM.
# ============================================================

from pathlib import Path
import json
import re
import unicodedata
import gc

import numpy as np
import polars as pl
from rapidfuzz import fuzz

# ------------------------------------------------------------
# Persistent paths
# ------------------------------------------------------------

DRIVE_PROJECT = Path("/content/drive/MyDrive/AMLC2026")

DRIVE_BLOCKING = (
    DRIVE_PROJECT / "cache" / "blocking"
)

DRIVE_ER_EDA = (
    DRIVE_PROJECT / "cache" / "state" / "er_eda"
)

TRAIN_DIR = DRIVE_PROJECT / "dataset" / "train"
TEST_DIR = DRIVE_PROJECT / "dataset" / "test"

files = {
    "train_source1": TRAIN_DIR / "train_source1.tsv",
    "train_source2": TRAIN_DIR / "train_source2.tsv",
    "train_source3": TRAIN_DIR / "train_source3.tsv",
    "train_ground_truth": TRAIN_DIR / "train_ground_truth.tsv",
    "test_source1": TEST_DIR / "test_source1.tsv",
    "test_source2": TEST_DIR / "test_source2.tsv",
    "test_source3": TEST_DIR / "test_source3.tsv",
}

# ------------------------------------------------------------
# Loader
# ------------------------------------------------------------

def clean_column_name(name: str) -> str:
    return str(name).replace("\ufeff", "").strip()


def scan_tsv(path: Path) -> pl.LazyFrame:
    lf = pl.scan_csv(
        path,
        separator="\t",
        infer_schema_length=1000,
        null_values=["", "null", "NULL", "None"],
    )

    schema = lf.collect_schema()

    rename_map = {
        col: clean_column_name(col)
        for col in schema.names()
        if clean_column_name(col) != col
    }

    if rename_map:
        lf = lf.rename(rename_map)

    return lf


# ------------------------------------------------------------
# Restore checkpoint artifacts
# ------------------------------------------------------------

artifact_names = [
    "gt",
    "gt_edges",
    "s1_eval",
    "eval_gt",
    "edge_eval",
    "per_s1_recall",
    "candidate_counts",
    "country_eval",
    "missed_edges",
    "missed_inspection",
]

for name in artifact_names:

    path = DRIVE_ER_EDA / f"{name}.parquet"

    if path.exists():
        globals()[name] = pl.read_parquet(path)

# ------------------------------------------------------------
# Restore normalized indexes
# ------------------------------------------------------------

S2_NAME_INDEX = DRIVE_BLOCKING / "train_s2_name_norm.parquet"
S3_NAME_INDEX = DRIVE_BLOCKING / "train_s3_name_norm.parquet"

assert S2_NAME_INDEX.exists()
assert S3_NAME_INDEX.exists()

s2_idx = pl.scan_parquet(S2_NAME_INDEX)
s3_idx = pl.scan_parquet(S3_NAME_INDEX)

# ------------------------------------------------------------
# Re-create the exact normalizer used by the notebook
# ------------------------------------------------------------

LEGAL_SUFFIX_PATTERN = (
    r"(?i)\b("
    r"private\s+limited|"
    r"private|"
    r"pvt\s+ltd|"
    r"pvt|"
    r"limited|"
    r"ltd|"
    r"corporation|"
    r"corp|"
    r"company|"
    r"co|"
    r"incorporated|"
    r"inc|"
    r"llc|"
    r"llp|"
    r"sarl|"
    r"sasu|"
    r"sas|"
    r"plc|"
    r"gmbh"
    r")\b"
)


def fast_name_expr(column="business_name"):

    return (
        pl.col(column)
        .fill_null("")
        .str.to_lowercase()
        .str.replace_all("&", " and ")
        .str.replace_all(
            r"[^\p{L}\p{N}\s]",
            " "
        )
        .str.replace_all(
            LEGAL_SUFFIX_PATTERN,
            " "
        )
        .str.replace_all(
            r"\s+",
            " "
        )
        .str.strip_chars()
    )


# ------------------------------------------------------------
# IMPORTANT FIX:
# Build the variable that Cell 33/35 expected.
# ------------------------------------------------------------

if "s1_norm_lookup" not in globals():

    s1_norm_lookup = (
        s1_eval
        .select([
            "entity_id",
            "business_name",
            "country",
        ])
        .rename({
            "entity_id": "source1_entity_id"
        })
        .with_columns(
            fast_name_expr().alias("name_norm")
        )
    )

# ------------------------------------------------------------
# Evaluation universe
#
# IMPORTANT:
# This remains the full 100k S1 evaluation population,
# INCLUDING the singleton S1s.
# ------------------------------------------------------------

EVAL_S1_IDS = s1_eval["entity_id"].to_list()

assert len(EVAL_S1_IDS) == 100_000

print("=" * 90)
print("AMLC 2026 — NEXT-STAGE STATE READY")
print("=" * 90)

print("Evaluation S1:", len(EVAL_S1_IDS))
print("Positive edges:", eval_gt.height)
print("S2 name index:", S2_NAME_INDEX)
print("S3 name index:", S3_NAME_INDEX)

print("\nRestored:")
for name in artifact_names:
    obj = globals().get(name)
    if obj is not None:
        print(f"  ✅ {name:22s} {obj.shape}")

print("\n✅ Ready for blocker-oracle analysis.")

AMLC 2026 — NEXT-STAGE STATE READY
Evaluation S1: 100000
Positive edges: 345980
S2 name index: /content/drive/MyDrive/AMLC2026/cache/blocking/train_s2_name_norm.parquet
S3 name index: /content/drive/MyDrive/AMLC2026/cache/blocking/train_s3_name_norm.parquet

Restored:
  ✅ gt                     (2206821, 2)
  ✅ gt_edges               (7638365, 3)
  ✅ s1_eval                (100000, 4)
  ✅ eval_gt                (345980, 3)
  ✅ edge_eval              (345980, 6)
  ✅ per_s1_recall          (94404, 4)
  ✅ candidate_counts       (100000, 5)
  ✅ country_eval           (345980, 6)
  ✅ missed_edges           (1000, 6)
  ✅ missed_inspection      (1000, 12)

✅ Ready for blocker-oracle analysis.


In [7]:
# ============================================================
# AMLC 2026 — CELL 47
# FULL POSITIVE-EDGE ORACLE
#
# Materializes only the 345,980 known positive evaluation edges.
# This is NOT candidate generation.
# It is used to measure blocker ceilings.
# ============================================================

print("=" * 90)
print("BUILDING POSITIVE-EDGE ORACLE")
print("=" * 90)

# ------------------------------------------------------------
# S1 records
# ------------------------------------------------------------

s1_truth = (
    s1_eval
    .select([
        pl.col("entity_id").alias("source1_entity_id"),
        pl.col("business_name").alias("s1_name"),
        pl.col("business_address").alias("s1_address"),
        pl.col("country").alias("s1_country"),
    ])
)

# ------------------------------------------------------------
# Positive S2 IDs
# ------------------------------------------------------------

s2_ids = (
    eval_gt
    .filter(pl.col("matched_source") == "S2")
    ["matched_entity_id"]
    .to_list()
)

s3_ids = (
    eval_gt
    .filter(pl.col("matched_source") == "S3")
    ["matched_entity_id"]
    .to_list()
)

print("S2 positive records:", len(s2_ids))
print("S3 positive records:", len(s3_ids))

# ------------------------------------------------------------
# Pull S2 truth records
# ------------------------------------------------------------

s2_truth = (
    scan_tsv(files["train_source2"])
    .filter(
        pl.col("entity_id").is_in(s2_ids)
    )
    .select([
        pl.col("entity_id").alias("matched_entity_id"),
        pl.col("business_name").alias("matched_name"),
        pl.col("business_address").alias("matched_address"),
        pl.col("country").alias("matched_country"),
    ])
    .collect()
)

# ------------------------------------------------------------
# Pull S3 truth records
# ------------------------------------------------------------

s3_truth = (
    scan_tsv(files["train_source3"])
    .filter(
        pl.col("entity_id").is_in(s3_ids)
    )
    .select([
        pl.col("entity_id").alias("matched_entity_id"),
        pl.col("business_name").alias("matched_name"),
        pl.col("business_address").alias("matched_address"),
        pl.col("country").alias("matched_country"),
    ])
    .collect()
)

truth_records = pl.concat([
    s2_truth,
    s3_truth,
])

# ------------------------------------------------------------
# Join everything
# ------------------------------------------------------------

positive_pairs = (
    eval_gt
    .join(
        s1_truth,
        on="source1_entity_id",
        how="left",
    )
    .join(
        truth_records,
        on="matched_entity_id",
        how="left",
    )
)

print("\nShape:", positive_pairs.shape)

print(
    positive_pairs
    .select([
        "source1_entity_id",
        "matched_entity_id",
        "matched_source",
        "s1_country",
        "matched_country",
        "s1_name",
        "matched_name",
        "s1_address",
        "matched_address",
    ])
    .head(10)
)

assert positive_pairs.height == 345_980

print("\n✅ Positive-edge oracle ready.")

BUILDING POSITIVE-EDGE ORACLE
S2 positive records: 167274
S3 positive records: 178706

Shape: (345980, 9)
shape: (10, 9)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ source1_e ┆ matched_e ┆ matched_s ┆ s1_countr ┆ … ┆ s1_name   ┆ matched_n ┆ s1_addres ┆ matched_ │
│ ntity_id  ┆ ntity_id  ┆ ource     ┆ y         ┆   ┆ ---       ┆ ame       ┆ s         ┆ address  │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ str       ┆ ---       ┆ ---       ┆ ---      │
│ str       ┆ str       ┆ str       ┆ str       ┆   ┆           ┆ str       ┆ str       ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ S1-869891 ┆ S3-274817 ┆ S3        ┆ India     ┆ … ┆ Laxmi     ┆ Laxmi Gbn ┆ New       ┆ New      │
│ 37        ┆ 120       ┆           ┆           ┆   ┆ Golden    ┆ lnvestmen ┆ Bridge    ┆ Bridge   │
│           ┆           ┆           ┆           ┆   ┆ Investmen ┆ ts   

In [10]:
# ============================================================
# AMLC 2026 — CELL 48
# EXACT VARIANT ORACLE
#
# Measures positive-edge recall ceilings for:
#   - native normalized name
#   - transliterated name
#   - normalized address
#   - transliterated address
#   - numeric address signature
#
# IMPORTANT:
# These are recall ceilings only.
# They tell us which blockers are worth indexing.
# ============================================================

from anyascii import anyascii

# ------------------------------------------------------------
# Python normalizers
# ------------------------------------------------------------

PY_SUFFIXES = {
    "private",
    "private limited",
    "pvt",
    "pvt ltd",
    "limited",
    "ltd",
    "corporation",
    "corp",
    "company",
    "co",
    "incorporated",
    "inc",
    "llc",
    "llp",
    "sarl",
    "sasu",
    "sas",
    "plc",
    "gmbh",
}


def normalize_ascii_name(text):

    if text is None:
        return ""

    x = anyascii(str(text))
    x = unicodedata.normalize("NFKC", x).casefold()

    x = x.replace("&", " and ")

    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()

    tokens = [
        t for t in x.split()
        if t not in PY_SUFFIXES
    ]

    return " ".join(tokens)


# Common address standardizations.
# Conservative enough for a blocker experiment.

ADDRESS_ABBR = [
    (r"\bstreet\b", " st "),
    (r"\broad\b", " rd "),
    (r"\bavenue\b", " ave "),
    (r"\bboulevard\b", " blvd "),
    (r"\bdrive\b", " dr "),
    (r"\blane\b", " ln "),
    (r"\bcourt\b", " ct "),
    (r"\bhighway\b", " hwy "),
    (r"\bparkway\b", " pkwy "),
    (r"\bapartment\b", " apt "),
    (r"\bsuite\b", " ste "),
    (r"\bbuilding\b", " bldg "),
]


def normalize_address_native(text):

    if text is None:
        return ""

    x = unicodedata.normalize(
        "NFKC",
        str(text)
    ).casefold()

    x = x.replace("&", " and ")

    x = re.sub(
        r"[^\w\s]",
        " ",
        x,
        flags=re.UNICODE,
    )

    for pattern, replacement in ADDRESS_ABBR:
        x = re.sub(
            pattern,
            replacement,
            x,
        )

    x = re.sub(r"\s+", " ", x).strip()

    return x


def normalize_address_ascii(text):

    if text is None:
        return ""

    x = anyascii(str(text))

    x = unicodedata.normalize(
        "NFKC",
        x
    ).casefold()

    x = x.replace("&", " and ")

    x = re.sub(
        r"[^a-z0-9\s]",
        " ",
        x,
    )

    for pattern, replacement in ADDRESS_ABBR:
        x = re.sub(
            pattern,
            replacement,
            x,
        )

    x = re.sub(r"\s+", " ", x).strip()

    return x


def numeric_signature(text):

    if text is None:
        return ""

    tokens = re.findall(
        r"\d+[a-zA-Z]?",
        str(text)
    )

    if not tokens:
        return ""

    return "|".join(
        sorted(set(tokens))
    )


# ------------------------------------------------------------
# Native name normalization using the notebook's exact logic
# ------------------------------------------------------------

positive_pairs = positive_pairs.with_columns([
    fast_name_expr("s1_name").alias("s1_name_native"),
    fast_name_expr("matched_name").alias("matched_name_native"),
])

# ------------------------------------------------------------
# Transliteration
# ------------------------------------------------------------

print("Computing transliterated name/address views...")

positive_pairs = positive_pairs.with_columns([
    pl.col("s1_name")
      .map_elements(
          normalize_ascii_name,
          return_dtype=pl.String,
          skip_nulls=False,
      )
      .alias("s1_name_ascii"),

    pl.col("matched_name")
      .map_elements(
          normalize_ascii_name,
          return_dtype=pl.String,
          skip_nulls=False,
      )
      .alias("matched_name_ascii"),

    pl.col("s1_address")
      .map_elements(
          normalize_address_native,
          return_dtype=pl.String,
          skip_nulls=False,
      )
      .alias("s1_addr_native"),

    pl.col("matched_address")
      .map_elements(
          normalize_address_native,
          return_dtype=pl.String,
          skip_nulls=False,
      )
      .alias("matched_addr_native"),

    pl.col("s1_address")
      .map_elements(
          normalize_address_ascii,
          return_dtype=pl.String,
          skip_nulls=False,
      )
      .alias("s1_addr_ascii"),

    pl.col("matched_address")
      .map_elements(
          normalize_address_ascii,
          return_dtype=pl.String,
          skip_nulls=False,
      )
      .alias("matched_addr_ascii"),

    pl.col("s1_address")
      .map_elements(
          numeric_signature,
          return_dtype=pl.String,
          skip_nulls=False,
      )
      .alias("s1_addr_nums"),

    pl.col("matched_address")
      .map_elements(
          numeric_signature,
          return_dtype=pl.String,
          skip_nulls=False,
      )
      .alias("matched_addr_nums"),
])

# ------------------------------------------------------------
# Exact conditions
# ------------------------------------------------------------

positive_pairs = positive_pairs.with_columns([

    (
        pl.col("s1_name_native")
        == pl.col("matched_name_native")
    ).alias("name_native_exact"),

    (
        (pl.col("s1_name_ascii") != "")
        & (
            pl.col("s1_name_ascii")
            == pl.col("matched_name_ascii")
        )
    ).alias("name_ascii_exact"),

    (
        (pl.col("s1_addr_native") != "")
        & (
            pl.col("s1_addr_native")
            == pl.col("matched_addr_native")
        )
    ).alias("address_native_exact"),

    (
        (pl.col("s1_addr_ascii") != "")
        & (
            pl.col("s1_addr_ascii")
            == pl.col("matched_addr_ascii")
        )
    ).alias("address_ascii_exact"),

    (
        (pl.col("s1_addr_nums") != "")
        & (
            pl.col("s1_addr_nums")
            == pl.col("matched_addr_nums")
        )
    ).alias("numeric_exact"),
])

# ------------------------------------------------------------
# Recall helper
# ------------------------------------------------------------

def oracle_summary(df, condition_cols):

    rows = []

    total = df.height

    for col in condition_cols:

        rows.append({
            "signal": col,
            "recovered_edges": int(
                df[col].sum()
            ),
            "edge_recall": float(
                df[col].mean()
            ),
        })

    return pl.DataFrame(rows).sort(
        "edge_recall",
        descending=True,
    )


signals = [
    "name_native_exact",
    "name_ascii_exact",
    "address_native_exact",
    "address_ascii_exact",
    "numeric_exact",
]

print("=" * 90)
print("POSITIVE-EDGE EXACT ORACLE")
print("=" * 90)

print(
    oracle_summary(
        positive_pairs,
        signals,
    )
)

print("\nBy source:")

for source in ["S2", "S3"]:

    print(f"\n--- {source} ---")

    print(
        oracle_summary(
            positive_pairs.filter(
                pl.col("matched_source") == source
            ),
            signals,
        )
    )

# ------------------------------------------------------------
# Cumulative unions
# ------------------------------------------------------------

positive_pairs = positive_pairs.with_columns([

    (
        pl.col("name_native_exact")
        | pl.col("name_ascii_exact")
    ).alias("name_union_exact"),

    (
        pl.col("name_native_exact")
        | pl.col("name_ascii_exact")
        | pl.col("address_native_exact")
        | pl.col("address_ascii_exact")
    ).alias("name_address_union_exact"),

    (
        pl.col("name_native_exact")
        | pl.col("name_ascii_exact")
        | pl.col("address_native_exact")
        | pl.col("address_ascii_exact")
        | pl.col("numeric_exact")
    ).alias("exact_union"),

])

print("\nCUMULATIVE")

print(
    oracle_summary(
        positive_pairs,
        [
            "name_union_exact",
            "name_address_union_exact",
            "exact_union",
        ],
    )
)

Computing transliterated name/address views...
POSITIVE-EDGE EXACT ORACLE
shape: (5, 3)
┌──────────────────────┬─────────────────┬─────────────┐
│ signal               ┆ recovered_edges ┆ edge_recall │
│ ---                  ┆ ---             ┆ ---         │
│ str                  ┆ i64             ┆ f64         │
╞══════════════════════╪═════════════════╪═════════════╡
│ numeric_exact        ┆ 200277          ┆ 0.578869    │
│ name_ascii_exact     ┆ 161460          ┆ 0.466674    │
│ name_native_exact    ┆ 144681          ┆ 0.418177    │
│ address_ascii_exact  ┆ 39930           ┆ 0.115411    │
│ address_native_exact ┆ 39717           ┆ 0.114796    │
└──────────────────────┴─────────────────┴─────────────┘

By source:

--- S2 ---
shape: (5, 3)
┌──────────────────────┬─────────────────┬─────────────┐
│ signal               ┆ recovered_edges ┆ edge_recall │
│ ---                  ┆ ---             ┆ ---         │
│ str                  ┆ i64             ┆ f64         │
╞══════════════════

In [11]:
# ============================================================
# AMLC 2026 — CELL 49
# FUZZY MISSED-POSITIVE TAXONOMY
#
# Only a sample is scored here.
# We do NOT run expensive RapidFuzz over the entire 346k edges.
# ============================================================

FUZZY_SAMPLE_N = 10_000

fuzzy_sample = (
    positive_pairs
    .filter(
        ~pl.col("name_union_exact")
    )
    .sample(
        n=min(
            FUZZY_SAMPLE_N,
            positive_pairs.filter(
                ~pl.col("name_union_exact")
            ).height,
        ),
        seed=2026,
    )
)

def safe_ratio(a, b):
    return fuzz.ratio(
        str(a),
        str(b),
    )

def safe_token_set(a, b):
    return fuzz.token_set_ratio(
        str(a),
        str(b),
    )


rows = []

for row in fuzzy_sample.iter_rows(named=True):

    name_native_ratio = safe_ratio(
        row["s1_name_native"],
        row["matched_name_native"],
    )

    name_ascii_ratio = safe_ratio(
        row["s1_name_ascii"],
        row["matched_name_ascii"],
    )

    name_token_ratio = safe_token_set(
        row["s1_name_ascii"],
        row["matched_name_ascii"],
    )

    addr_ascii_ratio = safe_ratio(
        row["s1_addr_ascii"],
        row["matched_addr_ascii"],
    )

    rows.append({
        "source1_entity_id":
            row["source1_entity_id"],

        "matched_entity_id":
            row["matched_entity_id"],

        "matched_source":
            row["matched_source"],

        "name_native_ratio":
            name_native_ratio,

        "name_ascii_ratio":
            name_ascii_ratio,

        "name_token_ratio":
            name_token_ratio,

        "address_ascii_ratio":
            addr_ascii_ratio,

        "name_ascii_exact":
            row["name_ascii_exact"],

        "address_ascii_exact":
            row["address_ascii_exact"],

        "numeric_exact":
            row["numeric_exact"],

        "s1_name":
            row["s1_name"],

        "matched_name":
            row["matched_name"],

        "s1_address":
            row["s1_address"],

        "matched_address":
            row["matched_address"],
    })

fuzzy_df = pl.DataFrame(rows)

print("=" * 90)
print("FUZZY MISSED-POSITIVE DISTRIBUTIONS")
print("=" * 90)

for col in [
    "name_native_ratio",
    "name_ascii_ratio",
    "name_token_ratio",
    "address_ascii_ratio",
]:

    print(f"\n{col}")

    print(
        fuzzy_df
        .select([
            pl.col(col).quantile(0.05).alias("p05"),
            pl.col(col).quantile(0.25).alias("p25"),
            pl.col(col).median().alias("p50"),
            pl.col(col).quantile(0.75).alias("p75"),
            pl.col(col).quantile(0.90).alias("p90"),
            pl.col(col).quantile(0.95).alias("p95"),
            pl.col(col).max().alias("max"),
        ])
    )

print("\nExamples — strongest transliterated name similarity among misses:")

print(
    fuzzy_df
    .sort("name_ascii_ratio", descending=True)
    .select([
        "matched_source",
        "name_ascii_ratio",
        "name_token_ratio",
        "s1_name",
        "matched_name",
        "s1_address",
        "matched_address",
    ])
    .head(30)
)

FUZZY MISSED-POSITIVE DISTRIBUTIONS

name_native_ratio
shape: (1, 7)
┌──────────┬───────────┬───────────┬──────┬───────────┬──────┬───────────┐
│ p05      ┆ p25       ┆ p50       ┆ p75  ┆ p90       ┆ p95  ┆ max       │
│ ---      ┆ ---       ┆ ---       ┆ ---  ┆ ---       ┆ ---  ┆ ---       │
│ f64      ┆ f64       ┆ f64       ┆ f64  ┆ f64       ┆ f64  ┆ f64       │
╞══════════╪═══════════╪═══════════╪══════╪═══════════╪══════╪═══════════╡
│ 4.651163 ┆ 60.606061 ┆ 77.777778 ┆ 87.5 ┆ 93.023256 ┆ 95.0 ┆ 98.701299 │
└──────────┴───────────┴───────────┴──────┴───────────┴──────┴───────────┘

name_ascii_ratio
shape: (1, 7)
┌──────┬──────┬───────────┬──────┬───────────┬───────────┬───────────┐
│ p05  ┆ p25  ┆ p50       ┆ p75  ┆ p90       ┆ p95       ┆ max       │
│ ---  ┆ ---  ┆ ---       ┆ ---  ┆ ---       ┆ ---       ┆ ---       │
│ f64  ┆ f64  ┆ f64       ┆ f64  ┆ f64       ┆ f64       ┆ f64       │
╞══════╪══════╪═══════════╪══════╪═══════════╪═══════════╪═══════════╡
│ 40.0 ┆ 65.0 ┆ 78.

In [12]:
# ============================================================
# AMLC 2026 — CELL 50
# BLOCK-KEY ORACLE
#
# Tests cheap retrieval keys WITHOUT building their indexes yet.
#
# Keys:
#   - native prefix/suffix
#   - transliterated prefix/suffix
#   - first / last token
#   - exact normalized address
#   - numeric signature
#
# We measure cumulative positive recall.
# ============================================================

# ------------------------------------------------------------
# Key expressions
# ------------------------------------------------------------

for prefix in [
    "s1_name_native",
    "matched_name_native",
    "s1_name_ascii",
    "matched_name_ascii",
]:

    positive_pairs = positive_pairs.with_columns([

        pl.col(prefix)
          .str.slice(0, 4)
          .alias(f"{prefix}_p4"),

        pl.col(prefix)
          .str.slice(-4)
          .alias(f"{prefix}_s4"),

        pl.col(prefix)
          .str.split(" ")
          .list.first()
          .fill_null("")
          .alias(f"{prefix}_first"),

        pl.col(prefix)
          .str.split(" ")
          .list.last()
          .fill_null("")
          .alias(f"{prefix}_last"),
    ])

# ------------------------------------------------------------
# Key matches
# ------------------------------------------------------------

positive_pairs = positive_pairs.with_columns([

    (
        (pl.col("s1_name_native_p4") != "")
        & (
            pl.col("s1_name_native_p4")
            == pl.col("matched_name_native_p4")
        )
    ).alias("native_prefix4"),

    (
        (pl.col("s1_name_native_s4") != "")
        & (
            pl.col("s1_name_native_s4")
            == pl.col("matched_name_native_s4")
        )
    ).alias("native_suffix4"),

    (
        (pl.col("s1_name_ascii_p4") != "")
        & (
            pl.col("s1_name_ascii_p4")
            == pl.col("matched_name_ascii_p4")
        )
    ).alias("ascii_prefix4"),

    (
        (pl.col("s1_name_ascii_s4") != "")
        & (
            pl.col("s1_name_ascii_s4")
            == pl.col("matched_name_ascii_s4")
        )
    ).alias("ascii_suffix4"),

    (
        (pl.col("s1_name_ascii_first") != "")
        & (
            pl.col("s1_name_ascii_first")
            == pl.col("matched_name_ascii_first")
        )
    ).alias("ascii_first_token"),

    (
        (pl.col("s1_name_ascii_last") != "")
        & (
            pl.col("s1_name_ascii_last")
            == pl.col("matched_name_ascii_last")
        )
    ).alias("ascii_last_token"),

])

# ------------------------------------------------------------
# Combined blocker unions
# ------------------------------------------------------------

positive_pairs = positive_pairs.with_columns([

    (
        pl.col("name_union_exact")
        | pl.col("native_prefix4")
        | pl.col("native_suffix4")
        | pl.col("ascii_prefix4")
        | pl.col("ascii_suffix4")
    ).alias("name_key_union"),

    (
        pl.col("name_union_exact")
        | pl.col("native_prefix4")
        | pl.col("native_suffix4")
        | pl.col("ascii_prefix4")
        | pl.col("ascii_suffix4")
        | pl.col("ascii_first_token")
        | pl.col("ascii_last_token")
    ).alias("name_plus_token_union"),

    (
        pl.col("name_union_exact")
        | pl.col("native_prefix4")
        | pl.col("native_suffix4")
        | pl.col("ascii_prefix4")
        | pl.col("ascii_suffix4")
        | pl.col("ascii_first_token")
        | pl.col("ascii_last_token")
        | pl.col("address_native_exact")
        | pl.col("address_ascii_exact")
        | pl.col("numeric_exact")
    ).alias("cheap_union"),

])

print("=" * 90)
print("BLOCK-KEY POSITIVE-EDGE ORACLE")
print("=" * 90)

print(
    oracle_summary(
        positive_pairs,
        [
            "name_union_exact",
            "name_key_union",
            "name_plus_token_union",
            "cheap_union",
        ],
    )
)

print("\nBy source:")

for source in ["S2", "S3"]:

    print(f"\n--- {source} ---")

    print(
        oracle_summary(
            positive_pairs.filter(
                pl.col("matched_source") == source
            ),
            [
                "name_union_exact",
                "name_key_union",
                "name_plus_token_union",
                "cheap_union",
            ],
        )
    )

print("\n✅ This is the decision gate for the next blocker implementation.")

BLOCK-KEY POSITIVE-EDGE ORACLE
shape: (4, 3)
┌───────────────────────┬─────────────────┬─────────────┐
│ signal                ┆ recovered_edges ┆ edge_recall │
│ ---                   ┆ ---             ┆ ---         │
│ str                   ┆ i64             ┆ f64         │
╞═══════════════════════╪═════════════════╪═════════════╡
│ cheap_union           ┆ 331342          ┆ 0.957691    │
│ name_plus_token_union ┆ 309611          ┆ 0.894881    │
│ name_key_union        ┆ 308063          ┆ 0.890407    │
│ name_union_exact      ┆ 161465          ┆ 0.466689    │
└───────────────────────┴─────────────────┴─────────────┘

By source:

--- S2 ---
shape: (4, 3)
┌───────────────────────┬─────────────────┬─────────────┐
│ signal                ┆ recovered_edges ┆ edge_recall │
│ ---                   ┆ ---             ┆ ---         │
│ str                   ┆ i64             ┆ f64         │
╞═══════════════════════╪═════════════════╪═════════════╡
│ cheap_union           ┆ 159149          ┆ 0.9

In [13]:
# ============================================================
# AMLC 2026 — CELL 51
# COMPOSITE BLOCKER ORACLE + MARGINAL CONTRIBUTION
#
# Goal:
#   1. Find which blocking-key combinations recover the
#      remaining positives without relying on single common keys.
#   2. Measure marginal contribution of each blocker.
#   3. Establish the exact residual-positive population that
#      the approximate retrieval stage must recover.
#
# IMPORTANT:
# This is still ORACLE ANALYSIS.
# We are NOT yet generating millions of candidate pairs.
# ============================================================

print("=" * 95)
print("COMPOSITE BLOCKER ORACLE")
print("=" * 95)

# ------------------------------------------------------------
# Base blocker conditions already produced in Cell 50
# ------------------------------------------------------------

base_blockers = {
    "native_name_exact":
        "name_native_exact",

    "ascii_name_exact":
        "name_ascii_exact",

    "numeric_exact":
        "numeric_exact",

    "address_native_exact":
        "address_native_exact",

    "address_ascii_exact":
        "address_ascii_exact",

    "native_prefix4":
        "native_prefix4",

    "native_suffix4":
        "native_suffix4",

    "ascii_prefix4":
        "ascii_prefix4",

    "ascii_suffix4":
        "ascii_suffix4",

    "ascii_first_token":
        "ascii_first_token",

    "ascii_last_token":
        "ascii_last_token",
}

# ------------------------------------------------------------
# Composite blockers
#
# These are much safer candidates for actual indexing because
# they combine signals instead of allowing one common token
# to explode the block.
# ------------------------------------------------------------

positive_pairs = positive_pairs.with_columns([

    # Transliteration-safe prefix + token
    (
        pl.col("ascii_prefix4")
        & pl.col("ascii_first_token")
    ).alias("ascii_prefix_first"),

    (
        pl.col("ascii_suffix4")
        & pl.col("ascii_last_token")
    ).alias("ascii_suffix_last"),

    (
        pl.col("ascii_first_token")
        & pl.col("ascii_last_token")
    ).alias("ascii_first_last"),

    # Native equivalents
    (
        pl.col("native_prefix4")
        & (
            pl.col("s1_name_native_first")
            == pl.col("matched_name_native_first")
        )
    ).alias("native_prefix_first"),

    (
        pl.col("native_suffix4")
        & (
            pl.col("s1_name_native_last")
            == pl.col("matched_name_native_last")
        )
    ).alias("native_suffix_last"),

    # Numeric + name anchors.
    # Much safer than numeric alone.
    (
        pl.col("numeric_exact")
        & pl.col("ascii_prefix4")
    ).alias("numeric_ascii_prefix"),

    (
        pl.col("numeric_exact")
        & pl.col("ascii_suffix4")
    ).alias("numeric_ascii_suffix"),

    (
        pl.col("numeric_exact")
        & pl.col("ascii_first_token")
    ).alias("numeric_ascii_first"),

    (
        pl.col("numeric_exact")
        & pl.col("ascii_last_token")
    ).alias("numeric_ascii_last"),

    # Address + name anchors.
    (
        pl.col("address_ascii_exact")
        & pl.col("ascii_first_token")
    ).alias("address_ascii_first"),

    (
        pl.col("address_ascii_exact")
        & pl.col("ascii_last_token")
    ).alias("address_ascii_last"),

    (
        pl.col("address_ascii_exact")
        & pl.col("ascii_prefix4")
    ).alias("address_ascii_prefix"),

    (
        pl.col("address_ascii_exact")
        & pl.col("ascii_suffix4")
    ).alias("address_ascii_suffix"),
])

# ------------------------------------------------------------
# Helper
# ------------------------------------------------------------

def recall_of(df, col):
    return float(df[col].mean())


def print_signal_table(df, signals, title):

    rows = []

    for signal in signals:

        rows.append({
            "signal": signal,
            "recovered_edges": int(df[signal].sum()),
            "edge_recall": float(df[signal].mean()),
        })

    out = (
        pl.DataFrame(rows)
        .sort(
            "edge_recall",
            descending=True,
        )
    )

    print(f"\n{title}")
    print("-" * 95)
    print(out)

    return out


composite_signals = [
    "ascii_prefix_first",
    "ascii_suffix_last",
    "ascii_first_last",
    "native_prefix_first",
    "native_suffix_last",

    "numeric_ascii_prefix",
    "numeric_ascii_suffix",
    "numeric_ascii_first",
    "numeric_ascii_last",

    "address_ascii_first",
    "address_ascii_last",
    "address_ascii_prefix",
    "address_ascii_suffix",
]

composite_summary = print_signal_table(
    positive_pairs,
    composite_signals,
    "COMPOSITE BLOCKER RECALL",
)

# ------------------------------------------------------------
# Marginal contribution analysis.
#
# We add blockers one at a time to an existing high-recall
# base and count newly recovered positives.
# ------------------------------------------------------------

# Start from exact normalized names.
current = (
    pl.col("name_union_exact")
)

marginal_rows = []

ordered_steps = [

    (
        "exact_name_union",
        pl.col("name_union_exact"),
    ),

    (
        "+ascii_prefix_first",
        pl.col("ascii_prefix_first"),
    ),

    (
        "+ascii_suffix_last",
        pl.col("ascii_suffix_last"),
    ),

    (
        "+ascii_first_last",
        pl.col("ascii_first_last"),
    ),

    (
        "+numeric_ascii_prefix",
        pl.col("numeric_ascii_prefix"),
    ),

    (
        "+numeric_ascii_suffix",
        pl.col("numeric_ascii_suffix"),
    ),

    (
        "+numeric_ascii_first",
        pl.col("numeric_ascii_first"),
    ),

    (
        "+numeric_ascii_last",
        pl.col("numeric_ascii_last"),
    ),

    (
        "+address_ascii_first",
        pl.col("address_ascii_first"),
    ),

    (
        "+address_ascii_last",
        pl.col("address_ascii_last"),
    ),

    (
        "+address_ascii_prefix",
        pl.col("address_ascii_prefix"),
    ),

    (
        "+address_ascii_suffix",
        pl.col("address_ascii_suffix"),
    ),

    # Keep the original cheap single-key signals at the end
    # so we can see exactly how much recall they add.
    (
        "+native_prefix4",
        pl.col("native_prefix4"),
    ),

    (
        "+native_suffix4",
        pl.col("native_suffix4"),
    ),

    (
        "+ascii_prefix4",
        pl.col("ascii_prefix4"),
    ),

    (
        "+ascii_suffix4",
        pl.col("ascii_suffix4"),
    ),

    (
        "+ascii_first_token",
        pl.col("ascii_first_token"),
    ),

    (
        "+ascii_last_token",
        pl.col("ascii_last_token"),
    ),

    (
        "+numeric_exact",
        pl.col("numeric_exact"),
    ),

    (
        "+address_ascii_exact",
        pl.col("address_ascii_exact"),
    ),
]

current_expr = pl.col("name_union_exact")

for step_name, step_expr in ordered_steps:

    previous_name = (
        f"__previous_{len(marginal_rows)}"
    )

    new_name = (
        f"__current_{len(marginal_rows)}"
    )

    if step_name == "exact_name_union":

        marginal_rows.append({
            "step": step_name,
            "new_edges": int(
                positive_pairs["name_union_exact"].sum()
            ),
            "cumulative_edges": int(
                positive_pairs["name_union_exact"].sum()
            ),
            "cumulative_recall": float(
                positive_pairs["name_union_exact"].mean()
            ),
        })

        continue

    previous_mask = current_expr

    current_expr = (
        current_expr
        | step_expr
    )

    tmp = positive_pairs.select([

        previous_mask.alias("__previous"),
        current_expr.alias("__current"),
    ])

    new_edges = int(
        (
            (~tmp["__previous"])
            & tmp["__current"]
        ).sum()
    )

    cumulative_edges = int(
        tmp["__current"].sum()
    )

    marginal_rows.append({
        "step": step_name,
        "new_edges": new_edges,
        "cumulative_edges": cumulative_edges,
        "cumulative_recall":
            cumulative_edges / positive_pairs.height,
    })


marginal_summary = pl.DataFrame(
    marginal_rows
)

print("\n")
print("=" * 95)
print("MARGINAL RECALL — ORDERED BLOCKER STACK")
print("=" * 95)

print(marginal_summary)

# ------------------------------------------------------------
# True residual after the full cheap union
# ------------------------------------------------------------

residual = (
    positive_pairs
    .filter(
        ~pl.col("cheap_union")
    )
)

print("\n")
print("=" * 95)
print("RESIDUAL POSITIVE EDGES")
print("=" * 95)

print(
    f"Residual edges: {residual.height:,}"
)

print(
    f"Residual fraction: "
    f"{residual.height / positive_pairs.height:.4%}"
)

print("\nBy source:")

print(
    residual
    .group_by("matched_source")
    .agg(
        pl.len().alias("residual_edges")
    )
    .with_columns(
        (
            pl.col("residual_edges")
            / positive_pairs.height
        ).alias("fraction_of_all_edges")
    )
    .sort("matched_source")
)

# ------------------------------------------------------------
# Residual by original S1 cardinality
# ------------------------------------------------------------

print("\nResidual by true S1 match cardinality:")

residual_cardinality = (
    residual
    .join(
        eval_gt
        .group_by("source1_entity_id")
        .agg(
            pl.len().alias("true_match_count")
        ),
        on="source1_entity_id",
        how="left",
    )
    .group_by("true_match_count")
    .agg(
        pl.len().alias("residual_edges")
    )
    .sort("true_match_count")
)

print(residual_cardinality)

print("\n✅ Cell 51 complete.")

COMPOSITE BLOCKER ORACLE

COMPOSITE BLOCKER RECALL
-----------------------------------------------------------------------------------------------
shape: (13, 3)
┌──────────────────────┬─────────────────┬─────────────┐
│ signal               ┆ recovered_edges ┆ edge_recall │
│ ---                  ┆ ---             ┆ ---         │
│ str                  ┆ i64             ┆ f64         │
╞══════════════════════╪═════════════════╪═════════════╡
│ ascii_prefix_first   ┆ 264883          ┆ 0.765602    │
│ native_prefix_first  ┆ 260330          ┆ 0.752442    │
│ ascii_suffix_last    ┆ 203902          ┆ 0.589346    │
│ native_suffix_last   ┆ 190237          ┆ 0.54985     │
│ ascii_first_last     ┆ 178870          ┆ 0.516995    │
│ …                    ┆ …               ┆ …           │
│ numeric_ascii_last   ┆ 118575          ┆ 0.342722    │
│ address_ascii_prefix ┆ 28917           ┆ 0.08358     │
│ address_ascii_first  ┆ 27274           ┆ 0.078831    │
│ address_ascii_suffix ┆ 24754          